# **SPRINT 08: Analítica de dades en Pandas**  



---
El sprint està organitzat en tres nivells de dificultat, on cada nivell aprofundeix més en:  

» Qualitat de les dades.  
» Capacitat de diagnòstic.  
» Tractament d'errors.  
» Enriquiment del dataset.

---

NIVELL 1:  
**Exercise 1. Exploració inicial del dataset**  
**Exercise 2. Validacions bàsiques de les dades i decisions d'eliminació**  
**Exercise 3. Transformacions necessàries i preparació del dataset**  

---


#>>>>> P2P <<<<<

---
---
---
> # **P2P Feedback:**  
(P2P with Albert, 14-SEP-2026)

>> **OVERALL OBSERVATIONS:**   
>> In general the feedback has been that I have risked imputing data values that should not necessary have been completed, but left as empty or unknown.  

>> - For example, my calculations to fill missing values for salaries (or filling 1 child for those that had 'te_fills' as True eventhough 'nombre_fills' showed 0, or imputating missing vehicle stats for those that have cars), although are calculated and explained following a solid logic and prior tests, the intial question of WHETHER these values should be completed/corrected was lost in my work. These imputations risk deviating possible future actions by the company, with repercussions on Salary Increases, for example, in levels 2 and 3 of the Sprint.  

>> - The transformations carried out on the 'genere' variable are very long-winded and laborious, and possibly I could have just extended the dictionary I used in the first phase of the corrections to include *all* name values, and apply the corrections globally in one action ... IF it was the correct decision to apply "standard perceived gender" per name as I did next (where I worked on correcting the most frequent gender per name, for the overall cleaning of what I deemed to be incorrect data entered).

>> - The calculated median values used to impute a single temperature value for each city in the dataset seem low (eg. Barcelona 16ºc), show check if negative values should be converted to NaN prior to calculating the medians to be used.

>> - It is good practise to separate where i have various actions to execute sharing a single cell in the notebook. As this makes it cleaner and easier to troubleshoot any errors in specific lines of code.

MY CONCLUSIONS:
>> I agree with Albert, and in hindsight think that I have got carried away with the possibilities of transformations that *could* be done (and the motivation of learning how to do each in Pandas), when I should have stepped-back on a number of occasions to first evaluate *whether they should actually have been done*.
---
---
---


# *NIVELL 1 - EDA estructurat i neteja bàsica*

data provided (saved to drive)

In [172]:
# /content/drive/MyDrive/Colab Notebooks/sprint8_complex.xlsx

# **NIVELL 1 - EXERCISE 1**  
---


### **Exploració inicial del dataset:**

**respondre, com a mínim, a les següents preguntes:**

» Quantes files i columnes té el dataset?  
» Quin tipus de dades conté cada columna?  
» Hi ha files duplicades?  
» Les dades semblen coherents a primera vista?  
» Detectes possibles problemes o inconsistències evidents?  

**Per fer-ho, hauràs de:**

» Analitzar les dimensions del dataset  
» Mostrar una mostra representativa de files  
» Revisar els tipus de variables  
» Les dades semblen coherents a primera vista?  
» Detectar files duplicades  
» Identificar i descriure possibles problemes inicials (sense solucionar-los encara)  

___


#### **Environment and notebook set-up:**

In [2]:
# upload file to read to my drive
# then,to give permission explicitly to access drive, need to run this code:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# importing libraries:

import pandas as pd
import numpy as np

In [4]:
# setting float format (2 decimal points) for whole notebook:
pd.options.display.float_format = '{:.2f}'.format

# setting downcasting message to NOT show (as got during changing column data types in the 'fills' section of code):
pd.set_option("future.no_silent_downcasting", True)

#### **Data load**

**Observations:**  
The original file provided is in **.xlsx** format, and direct import into Python results in numerous character encoding incompatibilities of which it would be laborious to identify the full extent and then sanitize in Python/Pandas.  

Therefore, I have chosen to **convert the original file provided as .csv format prior to importing into Python** (action that is not expressly prohibited in task instructions for this Sprint), as the .csv file format import permits the 'encoding' parameter (if needed) and allows for calibrating a much cleaner import to Python with minimum character encoding mismatches in the resultimg raw dataframe.  

Encoding parameters used:  
1) original .xlsx file saved as .csv with ISO 8859 encoding.  
2) read_csv (default encoder UTF8) used to import file for minimum character corruption.

In [5]:
# loading file that was saved as csv
df = pd.read_csv("/content/drive/MyDrive/sprint8_complex.csv", sep = ";")


» Mostrar una mostra representativa de files

In [6]:
df.head()

,Unnamed: 0,Nom,Cognoms,DNI,Pa�s d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,G�nere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.00,3.00,1958.00,Dona,"1,469 �,�",True,NaN,Grup B,Cap de projecte,3.00,True,32108.00,25.00,10.10
1,1,Marc,Muñoz,48840994W,Espanya,Alacant,8.00,11.00,1960.00,H,"2,718 �,�",NaN,True,Grup C,Senior analyst,0.00,True,19496.00,10.40,18.70
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.00,4.00,1961.00,D,1358 euros,True,NaN,Grup A,Tècnic IT,4.00,NaN,NaN,NaN,16.70
3,3,Pol,Gil,58586340F,Espanya,Sevilla,13.00,10.00,1985.00,H,"1,478 �,�",True,NaN,Grup B,Data Analyst,2.00,False,NaN,NaN,18.30
4,4,David,Vila,82070937P,Espanya,Bilbao,29.00,11.00,1965.00,Dona,"1,284 �,�",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.80,13.10


#### **» Quantes files i columnes té el dataset?:**  

» Analitzar les dimensions del dataset  
The imported raw dataset has 1007 rows and 20 columns.

In [7]:
df.shape

(1007, 20)

#### **» Quin tipus de dades conté cada columna?:**

The majority of columns are 'object' (12), followed by 'float' (7) and integer (1).  
Various datatypes will need modifying so that:  
-  those containing numbers have true numerical values (make sure either integer or float datatypes), thus permitting precise analysis of the numerical data later on in the project.  
- the columns representing a Date, or Boolean information, are converted into the corresponding datatype for error-free treatment in the analysis stage of the project.

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1007 entries, 0 to 1006
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  1007 non-null   int64  
 1   Nom                         1006 non-null   object 
 2   Cognoms                     1006 non-null   object 
 3   DNI                         1006 non-null   object 
 4   Pa�s d'origen               1006 non-null   object 
 5   Ciutat                      1006 non-null   object 
 6   Dia de Naixement            1006 non-null   float64
 7   Mes de Naixement            1006 non-null   float64
 8   Any de Naixement            1006 non-null   float64
 9   G�nere                      966 non-null    object 
 10  Salari mensual              1006 non-null   object 
 11  Fills                       419 non-null    object 
 12  No Fills                    585 non-null    object 
 13  Grup Professional           1005 

#### **» Hi ha files duplicades?:**

After inspection I see that there are **no complete rows duplicated**.  

When I exclude the first column, 'Unnamed: 0', from the selection (which suggests itself as a form of id for each row/employee) there *are* **4 rows flagged as 'partial' duplicates** (duplicate values are not present in all the columns).

On further investigation I see that there a total of **15 DNIs duplicated**, which will need to be looked at carefully to see if there are errors in the data or if there is an explanation that justifies the repetitions.  

Manual revision will be needed to check the exact situation for each case (keep, correct or eliminate), which should also correct the 4 duplicate rows identified previously.

In [9]:
# check if duplciated full rows... = NO
df.duplicated().sum()

np.int64(0)

In [10]:
# checking if duplicated rows if exclude the column 'Unnamed: 0': = YES, 4

### save column names excluding 'Unnamed: 0':
consider = [x for x in df.columns if x != "Unnamed: 0"]
### apply selection as a subset:
df_dupes_except_id = df.loc[df.duplicated(subset=consider, keep = False), consider]
### show count of rows after selection applied:
print(df_dupes_except_id.count())

# Show rows identified here as dupes:
print(f"\nDuplicate rows, excluding column[0]\n {df_dupes_except_id.sort_values(['DNI'])}")

Nom                           8
Cognoms                       8
DNI                           8
Pa�s d'origen                 8
Ciutat                        8
Dia de Naixement              8
Mes de Naixement              8
Any de Naixement              8
G�nere                        8
Salari mensual                8
Fills                         2
No Fills                      6
Grup Professional             8
Càrrec                        8
Nombre_fills                  8
Te_cotxe                      4
Km_anuals                     4
Consum_mitja_L_100km          4
Temperatura_mitjana_ciutat    8
dtype: int64

Duplicate rows, excluding column[0]
           Nom          Cognoms        DNI Pa�s d'origen    Ciutat  \
757     Oriol           Ferrer  52590001R       Espanya  Zaragoza   
1004    Oriol           Ferrer  52590001R       Espanya  Zaragoza   
36        Ida  Karlsen Eriksen  55283701C       Noruega      Oslo   
40        Ida  Karlsen Eriksen  55283701C       Noruega      Oslo

In [11]:
# checking if duplicated DNIs: = 15

df.duplicated(['DNI']).sum()

np.int64(15)

#### **» Les dades semblen coherents a primera vista?:**

The data appears generally coherent, although much cleaning is needed before a detailed inspection can be carried out in order to determine coherence of individual records.  
This data cleaning and preparation ready for analysis will include:  
- evaluation of duplicate records identified previously.
- standarization of categorical variables and datatype formats throughout  
- detection of outliers or impossible values in the numerical variables
- evaluation of missing data, and if it is recuperable or invalidates the whole record  
- coherence between specific related variables (eg 'te_fills' and 'nombre_fills', or average temperatures for the same city in 'temperatura_mitjana_ciutat', etc).

In [12]:
# overview of basic ranges and values of numerical variables:
df.describe()

,Unnamed: 0,Dia de Naixement,Mes de Naixement,Any de Naixement,Nombre_fills,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
count,1007.00,1006.00,1006.00,1006.00,709.00,315.00,261.00,936.00
mean,503.00,15.03,8.56,2002.23,1.20,55951.05,11.02,15.38
std,290.84,10.33,14.30,143.12,1.39,62095.44,8.78,7.28
min,0.00,-99.00,0.00,1940.00,0.00,-50.00,0.50,-10.00
25%,251.50,7.00,3.25,1969.00,0.00,22632.00,4.70,12.80
50%,503.00,15.00,6.00,1982.00,1.00,56135.00,9.80,15.50
75%,754.50,24.00,10.00,1995.00,2.00,80000.00,13.60,18.30
max,1006.00,32.00,99.00,3000.00,5.00,1000000.00,25.00,42.00


#### **» Detectes possibles problemes o inconsistències evidents?:**

- Initial character encoding inconsistencies from importing the original file will need to be corrected (letters with accents in column headers, the euro symbol in the salary column, etc).  
- Consider whether column 0 should be the index (would change original order) or whether preferable to use as an "employee id" column to assign a unique identifier to each record/employee.  
- The 'genere' column has 9 unique values. These should be verified and decisions made on the values that are accepted for this variable.  
- Job titles are not normalised (check other categorical columns also)  
- Extreme values will need to be checked and a decision made (eg birth year 3000, annual km 1M, negative values for km or temperature, etc.).  
- Evaluation is needed regarding duplicate or missing values (eg. salaries, DNIs etc.):  
  - Decide whether to impute them or if they are considered irrecuperable.  
  - If deemed irrecuperable, decide in each case whether it is prefereable to discard the row or to mark the value as NaN.  
  - Consider the value of the data that is present in the rest of the record when making this decision.  
  - Also evaluate the reliability of predicting the specific missing value and the overall consequences this may have in the analysis stage of the project.  

- As stated above, further investigation is needed into the coherence of data in the various columns relating to numbers of children, for example, or vehicle statistics.  



In [13]:
# character encoding needs fixing:
df.columns

Index(['Unnamed: 0', 'Nom', 'Cognoms', 'DNI', 'Pa�s d'origen', 'Ciutat',
       'Dia de Naixement', 'Mes de Naixement', 'Any de Naixement', 'G�nere',
       'Salari mensual', 'Fills', 'No Fills', 'Grup Professional', 'Càrrec',
       'Nombre_fills', 'Te_cotxe', 'Km_anuals', 'Consum_mitja_L_100km',
       'Temperatura_mitjana_ciutat'],
      dtype='object')

In [14]:
df.describe(exclude = "number")

# here can see that the catergorical column, Genere, has 9 unique values...

,Nom,Cognoms,DNI,Pa�s d'origen,Ciutat,G�nere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Te_cotxe
count,1006,1006,1006,1006,1006,966,1006,419,585,1005,1005,658
unique,103,468,991,6,45,9,855,1,2,5,14,2
top,Adrià,Ferrer,90585882A,Espanya,Palma,D,n/d,True,True,Grup A,Data Analyst,False
freq,34,22,6,719,58,325,14,419,584,466,142,330


In [15]:
# missing data:

null_per_column = df.isnull().sum()
null_per_column

,0
Unnamed: 0,0
Nom,1
Cognoms,1
DNI,1
Pa�s d'origen,1
Ciutat,1
Dia de Naixement,1
Mes de Naixement,1
Any de Naixement,1
G�nere,41


# **NIVELL 1 - EXERCISE 2**  
---


### **Validacions bàsiques de les dades i decisions d'eliminació**

En aquest exercici hauràs de validar aspectes clau del dataset que són crítics en qualsevol context real d'anàlisi de dades, prenent decisions justificades sobre la validesa o eliminació de registres.   

**Aspectes a validar:**  

- » Existeixen identificadors únics (DNI) per a cada persona?  
- » Què faràs amb les files duplicades o registres repetits?  
- » Hi ha valors obligatoris que falten o informació incoherent?  
- » Detectes formats incorrectes, errors de codificació o valors impossibles?  

**Concretament, hauràs de:**  

- » Corregir possibles problemes de codificació i accentuació en els noms de les
columnes i en els valors de text, si escau. **(->> COLUMN FIXES)**  
- » Detectar i eliminar files duplicades exactes, indicant quants registres s'han eliminat.  **(->> DNI FIXES)**  
- » Comprovar la unicitat del DNI. En cas de detectar DNIs duplicats, hauràs d'analitzar la coherència de la informació associada i descartar o anul·lar els registres que no siguin fiables, justificant clarament la decisió presa. **(->> DNI FIXES)**   
- » Identificar els camps clau que no haurien de contenir valors nuls, detectar registres amb informació insuficient i eliminar únicament aquelles files completament buides o amb informació irrecuperable o no substituïble, explicant per què no és possible ni raonable imputar-les.  **(->> EXERCISE 3)**  


---



***note on the following Exercise 2 instructions:***  
- *"» Hi ha valors obligatoris que falten o informació incoherent?"*  
- *"» Detectes formats incorrectes, errors de codificació o valors impossibles"*   
- *"Concretament hauras de: » Identificar els camps clau que no haurien de contenir valors nuls, detectar registres amb informació insuficient i eliminar únicament aquelles files completament buides o amb informació irrecuperable o no substituïble, explicant per què no és possible ni raonable imputar-les."*


I have **included my investigations and actions regarding these points of the EDA project in Exercise 3**, as each column required careful study and specific tests to check coherence, validity, the consequences and importance of missing values etc.  

In exercise 3, below, I systematically assess the data column by column to decide the optimum actions to be taken in each case in order to preserve reliability of the data, assure coherence between series and achieve the maximum analytical value without corrupting authenticity of the original data.


## COLUMN FIXES

### Fixing column headers:

In [16]:
# OUTPUT = original column names (before fixing):

df.columns

Index(['Unnamed: 0', 'Nom', 'Cognoms', 'DNI', 'Pa�s d'origen', 'Ciutat',
       'Dia de Naixement', 'Mes de Naixement', 'Any de Naixement', 'G�nere',
       'Salari mensual', 'Fills', 'No Fills', 'Grup Professional', 'Càrrec',
       'Nombre_fills', 'Te_cotxe', 'Km_anuals', 'Consum_mitja_L_100km',
       'Temperatura_mitjana_ciutat'],
      dtype='object')

In [17]:
# fixing column titles...

df.columns = (df.columns
		.str.strip()
		.str.lower()
		.str.replace(' ', '_')
    .str.replace('�', 'i')
    .str.replace('ginere', 'genere')
    .str.replace('dni', 'DNI'))

df.columns

Index(['unnamed:_0', 'nom', 'cognoms', 'DNI', 'pais_d'origen', 'ciutat',
       'dia_de_naixement', 'mes_de_naixement', 'any_de_naixement', 'genere',
       'salari_mensual', 'fills', 'no_fills', 'grup_professional', 'càrrec',
       'nombre_fills', 'te_cotxe', 'km_anuals', 'consum_mitja_l_100km',
       'temperatura_mitjana_ciutat'],
      dtype='object')

### Col 0: Deciding procedure for unnamed column 0 (make index or make employee_id) and implementing

Need to decide how to treat the first column, **'Unnamed: 0'**, as it suggests itself as a registration or original index number from when each record was opened, which could offer valuable chronological information between the various data records which is not present in any of the other columns.  
This aspect could be especially useful during investigation of duplicate rows and DNIs to determine which should be kept and which discarded.  
OPTIONS:  
-  use 'Unnamed: 0', as the index (reset_index)  
-  maintain 'Unnamed: 0', renaming it as 'employee id'. In this case, consider:   
    - how best to replace ID '0'?  
      - jog all 1 value, so is employeed_id 1?  
      - or add at end, as employeed_id 1007?


Checking information valid in first record (i.e. to be kept), that would be "employee_id: 0"

In [18]:
# checking information valid in first record, that would be "employee_id: 0"...
df.loc[0, : ]

,0
unnamed:_0,0
nom,Joana
cognoms,Gil Navarro
DNI,66722344X
pais_d'origen,Espanya
ciutat,Valladolid
dia_de_naixement,23.00
mes_de_naixement,3.00
any_de_naixement,1958.00
genere,Dona


Need to verify no duplicate values in the column, which would be incompatible with using either as the index or as employee_id.

In [19]:
df.duplicated(['unnamed:_0']).sum()

np.int64(0)

Will keep this data as a column and rename ir employee_id.  
Current employee_id 0 (Joana Gil Navarro) will become employee_id 1007.

In [20]:
# applying the rename
df = df.rename(columns={'unnamed:_0' : 'employee_id'})


In [21]:
# modifiying employee_id 0 to employee_id 1007
df.iat[0,0] = 1007


In [22]:
# checking column rename and employee_id change:
df.head(3)

,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
0,1007,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.00,3.00,1958.00,Dona,"1,469 �,�",True,NaN,Grup B,Cap de projecte,3.00,True,32108.00,25.00,10.10
1,1,Marc,Muñoz,48840994W,Espanya,Alacant,8.00,11.00,1960.00,H,"2,718 �,�",NaN,True,Grup C,Senior analyst,0.00,True,19496.00,10.40,18.70
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.00,4.00,1961.00,D,1358 euros,True,NaN,Grup A,Tècnic IT,4.00,NaN,NaN,NaN,16.70


### checking normalization of categorical columns

In order to verify the values that have been used in the various categorical columns ("pais_d'origen", "ciutat", "càrrec"), need to:  
- list unique values recorded in each column  
- decide if any are:  
      - ambiguous (same concept recorded with differing values)  
      - or contain codification errors (typos or empty)

In [23]:
# get full list of columns:
df.columns

# save list of categorical columns to check values normalised:
categorical = ["pais_d'origen", "ciutat", "càrrec"]

#printing list of each, to scan manually to check if values normalised or not...
for col in categorical:
  print(f"\nvalues in column: {col}")
  listed_values = df[col].dropna().unique().tolist() # as have changed order, now need to exclude nulls as not yet removed!
  listed_values.sort()
  for each in listed_values:
    print(each)



values in column: pais_d'origen
Alemanya
Espanya
França
Itàlia
Narnia
Noruega

values in column: ciutat
A Coruña
Alacant
Barcelona
Bergen
Berlin
Bilbao
Bologna
Bordeaux
Desconeguda
Düsseldorf
Firenze
Frankfurt
Genova
Girona
Hamburg
Köln
Las Palmas de Gran Canaria
Leipzig
Lille
Lyon
Madrid
Marseille
Milano
Murcia
Màlaga
München
Nantes
Napoli
Nice
Oslo
Palma
Paris
Roma
Sevilla
Stavanger
Stuttgart
Tarragona
Torino
Toulouse
Tromsø
Trondheim
Valladolid
València
Venezia
Zaragoza

values in column: càrrec
???
Admin.
Administratiu
Analista
Analista junior
Analyst
Cap de Projecte
Cap de projecte
Data Analyst
Data analyst
Project lead
Responsable de vendes
Senior analyst
Tècnic IT


FINDINGS:  
Are the categorical columns normalised?

"pais_d'origen" = ok (but check value 'Narnia' identified)  
'ciutat' = ok (but check value 'Desconeguda' identified)  
'càrrec' = NEEDS CLEANING

While checking unique values in the categorical variables, identified a row with dummy data. Noted for elimination.

In [24]:
print(df.loc[(df['ciutat'] == 'Desconeguda')])


      employee_id  nom cognoms   DNI pais_d'origen       ciutat  \
1006         1006  ???     ###  XXXX        Narnia  Desconeguda   

      dia_de_naixement  mes_de_naixement  any_de_naixement genere  \
1006            -99.00             99.00           3000.00     ??   

     salari_mensual fills no_fills grup_professional càrrec  nombre_fills  \
1006          molts   NaN      NaN                 Z    ???           NaN   

     te_cotxe  km_anuals  consum_mitja_l_100km  temperatura_mitjana_ciutat  
1006      NaN        NaN                   NaN                         NaN  


## DNI FIXES

As established in the initial study in Exercise 1, regarding duplicate rows, the cases identified are tied to duplicate DNIs, so establishing DNI uniqueness will correct duplicate rows in the dataset.  

 I did an exhaustive manual study of the employee records that either had the same DNIs or were missing this information, to determine case by case which rows were valid, which needed modification, or which should be eliminated.

In [25]:
# forcing showing all columns (so full width and all columns):

pd.options.display.max_columns = None

In [26]:
# checking DNIs that are duplicates, to decide fixes and apply:

duplicate_rows = df[df.duplicated(subset=['DNI'], keep=False)]
print(f"{len(duplicate_rows)} rows are duplictae DNIs \n")

display(duplicate_rows.sort_values('DNI'))

24 rows are duplictae DNIs 



,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
604,604,Laia,Navarro,15909601K,Espanya,Bilbao,30.00,5.00,1958.00,D,"1,676 �,�",True,False,Grup A,Analista,1.00,True,50.00,NaN,14.20
250,250,Laia,Navarro Vila,15909601K,Espanya,Bilbao,30.00,5.00,1958.00,D,1235 euros,True,True,Grup D,Analista junior,0.00,False,NaN,NaN,10.30
759,759,Oriol,Gómez,16618423V,Espanya,Sevilla,10.00,1.00,1968.00,NaN,"2,000 �,�",NaN,NaN,Grup A,Responsable de vendes,5.00,NaN,NaN,NaN,20.70
758,758,Oriol,Gómez,16618423V,Espanya,Màlaga,10.00,1.00,1968.00,H,1879 euros,True,NaN,Grup B,Data Analyst,4.00,False,NaN,NaN,NaN
754,754,Oriol,Gómez,16618423V,Espanya,Barcelona,10.00,1.00,1968.00,Home,"2.358 �,�",True,NaN,NaN,NaN,4.00,NaN,NaN,NaN,NaN
654,654,Oriol,Gómez,16618423V,Espanya,Madrid,10.00,1.00,1968.00,Home,1731 euros,True,NaN,Grup B,Data analyst,4.00,NaN,NaN,NaN,NaN
692,692,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.00,10.00,1981.00,Dona,"1663�,�",NaN,True,Grup C,Cap de projecte,NaN,True,80000.00,10.70,16.90
558,558,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.00,10.00,1981.00,Dona,"1,176 �,�",NaN,True,Grup A,Data Analyst,0.00,False,NaN,NaN,38.00
757,757,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.00,7.00,1973.00,D,"786�,�",NaN,True,Grup A,Analista,0.00,True,81381.00,6.50,13.90
1004,1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.00,7.00,1973.00,D,"786�,�",NaN,True,Grup A,Analista,0.00,True,81381.00,6.50,13.90


FINDINGS:  
There are duplicate DNIs that are due to either promotions or changes to city annoted. Some are missing values (a few to delete) and for the rest, I will keep the most recent (by employee_id, or grup_professional, depending on the case).    
At the end of the section I have a list of DNIs that are missing to be given to HR with instructions that they contact these employees or their managers to complete the missing data. These rows are not deleted as there is other valuable data in their records that should not be removed from the analysis.


**Summary of cases identified and action taken:**  

KEEP HIGHEST ID (but copy some column data first)
- Oriol	Gómez	16618423V	 delete 3, keep most recent/highest grup_professional (A) (copy first genere, salary and fills from 754)

KEEP ROW WITH HIGHEST employee_id (promotions etc):
- Laia	Navarro	15909601K	, promoted, but check info in original row not in newest row = ok!
- Alexia	Pérez Díaz	49529438G	pormoted, keep most recent by employee_id (but grup_professional lower??) = ok!
- Sara Serra 90585882A, keep most recent (wage rise) as she is probs this DNI as has 3 rows with it (DNI which has 4 different employees recorded..replace others with NaN FIRST = OK!!)

KEEP ROW LOWEST employee_id (keep original row):
- Oriol	Ferrer	52590001R simple duplictaion, same data (keep oldest by employee_id)
- Ida	Karlsen Eriksen	55283701 simple duplictaion, same data (keep oldest by employee_id)
- Claudia	Gil Ferrer	55297384F simple duplictaion, same data (keep oldest by employee_id)
- Alba	Muñoz	97954019T simple duplictaion, same data (keep oldest by employee_id)

INDIVIDUAL CASES:  
- MArta Ruiz and Guillem Pujol COsta , delete 1? put both NaN??
- checking guillem pujol costa  (ONLY ONE) and marta ruiz (ALSO ONLY ONE WITH COINCIDING DOB/POSITION ETC)... >>> which to keep?
- 90585882A...  DNI which has 4 different employees recorded...
    - Sara Serra,  keep most recent (wage rise) as noted above
    - put NaN: Oriol	Vila Casals... only one, put NaN
    - put NaN: Pol	Serra	... theres another one same name/surname, but NO other data coincides (city, age, DOB, position, etc), put NaN
    - put NaN: Xavier	Vidal... only one, put NaN

### Fixing case Oriol Gomez

Oriol Gómez 16618423V delete 3, keep most recent/highest grup_professional (A).  
Copy first genere, salary and fills from 754, which has many other values as NaN (increase to wage and info on children is data that shouldnt be lost).

In [27]:
# retrieving data on Oriol Gomez, to check before deleting dupes (= keep record 759)
df.loc[df['DNI'] == '16618423V'].sort_values('employee_id', ascending= False)

,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
759,759,Oriol,Gómez,16618423V,Espanya,Sevilla,10.00,1.00,1968.00,NaN,"2,000 �,�",NaN,NaN,Grup A,Responsable de vendes,5.00,NaN,NaN,NaN,20.70
758,758,Oriol,Gómez,16618423V,Espanya,Màlaga,10.00,1.00,1968.00,H,1879 euros,True,NaN,Grup B,Data Analyst,4.00,False,NaN,NaN,NaN
754,754,Oriol,Gómez,16618423V,Espanya,Barcelona,10.00,1.00,1968.00,Home,"2.358 �,�",True,NaN,NaN,NaN,4.00,NaN,NaN,NaN,NaN
654,654,Oriol,Gómez,16618423V,Espanya,Madrid,10.00,1.00,1968.00,Home,1731 euros,True,NaN,Grup B,Data analyst,4.00,NaN,NaN,NaN,NaN


In [28]:
# fix copying data from other rows before deleting
df.loc[df['employee_id'] == 759, ('genere', 'salari_mensual', 'fills')] = ('H','2.358', True)

In [29]:
# checking results of data copied to original df:
df.loc[df['DNI'] == '16618423V'].sort_values('employee_id', ascending= False)

,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
759,759,Oriol,Gómez,16618423V,Espanya,Sevilla,10.00,1.00,1968.00,H,2.358,True,NaN,Grup A,Responsable de vendes,5.00,NaN,NaN,NaN,20.70
758,758,Oriol,Gómez,16618423V,Espanya,Màlaga,10.00,1.00,1968.00,H,1879 euros,True,NaN,Grup B,Data Analyst,4.00,False,NaN,NaN,NaN
754,754,Oriol,Gómez,16618423V,Espanya,Barcelona,10.00,1.00,1968.00,Home,"2.358 �,�",True,NaN,NaN,NaN,4.00,NaN,NaN,NaN,NaN
654,654,Oriol,Gómez,16618423V,Espanya,Madrid,10.00,1.00,1968.00,Home,1731 euros,True,NaN,Grup B,Data analyst,4.00,NaN,NaN,NaN,NaN


In [30]:
# DELETING DUPLICATE ORIOL GOMEZ ROWS FROM MAIN DF:
df.drop([758, 754, 654], inplace = True)
# to check deleted: = OK
df.loc[df['DNI'] == '16618423V']

,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
759,759,Oriol,Gómez,16618423V,Espanya,Sevilla,10.00,1.00,1968.00,H,2.358,True,NaN,Grup A,Responsable de vendes,5.00,NaN,NaN,NaN,20.70


### DNIs guillem pujol costa & marta ruiz (fixed below with Nan)

Marta Ruiz and Guillem Pujol Costa , have the same DNI.  
After checking rest of data for each, and if duplicate records with same names, have decided to put NaN rather than delete employees, and pass list for follow-up by HR team to contact and ammend all DNIs that show NaN at end of cleaning.

In [31]:
# checking guillem pujol costa  (ONLY ONE WITH SAME NAME/SURNAME)
display(df.loc[(df['cognoms'] == 'Pujol Costa') & (df['nom'] == 'Guillem')])
display(df.loc[(df['cognoms'] == 'Pujol Costa') ])

# checking marta ruiz (ALSO ONLY ONE WITH COINCIDING DOB/POSITION ETC)
display(df.loc[(df['cognoms'].str.startswith('Ruiz')) & (df['nom'] == 'Marta')])


,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
89,89,Guillem,Pujol Costa,54013571W,Espanya,A Coruña,17.00,7.00,1983.00,D,"1.574 �,�",True,NaN,Grup B,Project lead,4.00,False,NaN,NaN,14.60


,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
89,89,Guillem,Pujol Costa,54013571W,Espanya,A Coruña,17.00,7.00,1983.00,D,"1.574 �,�",True,NaN,Grup B,Project lead,4.00,False,NaN,NaN,14.60
427,427,David,Pujol Costa,86758847Y,Espanya,Zaragoza,6.00,1.00,1969.00,D,"1,644 �,�",NaN,True,Grup B,Analista,NaN,True,21422.00,9.80,16.70


,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
65,65,Marta,Ruiz Puig,29602406G,Espanya,Madrid,11.00,11.00,2003.00,F,1664 euros,NaN,True,Grup B,Analista,NaN,True,80000.00,25.00,13.90
142,142,Marta,Ruiz,54013571W,Espanya,Palma,24.00,5.00,1982.00,H,"1,276 �,�",True,True,Grup A,Analista,0.00,False,NaN,NaN,15.70
370,370,Marta,Ruiz,70552489Z,Espanya,Las Palmas de Gran Canaria,14.00,6.00,1997.00,H,"1,317 �,�",True,NaN,Grup A,Tècnic IT,2.00,False,NaN,NaN,22.40


### fixing DNIs need to change to NaN  



This section included checking the DNI with 4 different employees recorded (notes above: keep latest Sara Serra and change rest to DNI NaN...), and individual checks on various employee names to see if they had more than 1 row (egs below for Oriol Vila CAsals and Xavier Vidal)  

As the deleting would be done by index row number, I re-ran the list of duplicate DNIs to double-check I had the current index rows.  

List of rows to put NaN in DNI:
- Marta Ruiz (index 142)
- Guillem Pujol Costa (index 89)
- Oriol	Vila Casals (index 104)
- Pol	Serra (index 114)
- Xavier	Vidal (index 281)

In [32]:
# display(df.loc[(df['cognoms'] == 'Vila Casals') & (df['nom'] == 'Oriol')])
display(df.loc[(df['cognoms'] == 'Vidal') & (df['nom'] == 'Xavier')])

,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
281,281,Xavier,Vidal,90585882A,Espanya,Alacant,2.00,7.00,1958.00,H,1179 euros,NaN,NaN,Grup A,Tècnic IT,0.00,True,63574.00,6.00,16.80


In [33]:
# re-running list of duplicate DNIs so assure correct INDEX numbers for each (after previous fixes carried out)
duplicate_rows = df[df.duplicated(subset=['DNI'], keep=False)]
print(f"{len(duplicate_rows)} rows are duplictae DNIs \n")

display(duplicate_rows.sort_values('DNI'))


20 rows are duplictae DNIs 



,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
250,250,Laia,Navarro Vila,15909601K,Espanya,Bilbao,30.00,5.00,1958.00,D,1235 euros,True,True,Grup D,Analista junior,0.00,False,NaN,NaN,10.30
604,604,Laia,Navarro,15909601K,Espanya,Bilbao,30.00,5.00,1958.00,D,"1,676 �,�",True,False,Grup A,Analista,1.00,True,50.00,NaN,14.20
558,558,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.00,10.00,1981.00,Dona,"1,176 �,�",NaN,True,Grup A,Data Analyst,0.00,False,NaN,NaN,38.00
692,692,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.00,10.00,1981.00,Dona,"1663�,�",NaN,True,Grup C,Cap de projecte,NaN,True,80000.00,10.70,16.90
1004,1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.00,7.00,1973.00,D,"786�,�",NaN,True,Grup A,Analista,0.00,True,81381.00,6.50,13.90
757,757,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.00,7.00,1973.00,D,"786�,�",NaN,True,Grup A,Analista,0.00,True,81381.00,6.50,13.90
89,89,Guillem,Pujol Costa,54013571W,Espanya,A Coruña,17.00,7.00,1983.00,D,"1.574 �,�",True,NaN,Grup B,Project lead,4.00,False,NaN,NaN,14.60
142,142,Marta,Ruiz,54013571W,Espanya,Palma,24.00,5.00,1982.00,H,"1,276 �,�",True,True,Grup A,Analista,0.00,False,NaN,NaN,15.70
36,464,Ida,Karlsen Eriksen,55283701C,Noruega,Oslo,2.00,9.00,1992.00,D,"1,614 �,�",True,NaN,Grup B,Analista,3.00,NaN,NaN,NaN,10.60
40,1001,Ida,Karlsen Eriksen,55283701C,Noruega,Oslo,2.00,9.00,1992.00,D,"1,614 �,�",True,NaN,Grup B,Analista,3.00,NaN,NaN,NaN,10.60


In [34]:
# import numpy to use np.nan...
import numpy as np

# verified index numbers of the rows to change to NaN
rows_to_NaN = [142, 89, 104, 114, 281]

# apply change to NaN for selected rows  in df:
df.loc[rows_to_NaN, 'DNI'] = np.nan


In [35]:
# checking worked
display(df.loc[rows_to_NaN])

,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
142,142,Marta,Ruiz,NaN,Espanya,Palma,24.00,5.00,1982.00,H,"1,276 �,�",True,True,Grup A,Analista,0.00,False,NaN,NaN,15.70
89,89,Guillem,Pujol Costa,NaN,Espanya,A Coruña,17.00,7.00,1983.00,D,"1.574 �,�",True,NaN,Grup B,Project lead,4.00,False,NaN,NaN,14.60
104,104,Oriol,Vila Casals,NaN,Espanya,Madrid,30.00,9.00,1964.00,H,1987 euros,NaN,True,Grup B,Responsable de vendes,0.00,False,NaN,NaN,13.50
114,114,Pol,Serra,NaN,Espanya,Girona,4.00,12.00,1960.00,H,"1,178 �,�",NaN,True,Grup A,Data Analyst,NaN,False,NaN,NaN,16.20
281,281,Xavier,Vidal,NaN,Espanya,Alacant,2.00,7.00,1958.00,H,1179 euros,NaN,NaN,Grup A,Tècnic IT,0.00,True,63574.00,6.00,16.80


### fixing keep highest (newest) and keep lowest (oldest) by index number:

As per notes above, identifying employees for each solution required:  

KEEP ROW WITH HIGHEST employee_id (promotions etc):
- Laia	Navarro	15909601K	(drop index 250)  >> COPY 2ND SURNAME TO LATEST ROW FIRST = OK
- Alexia	Pérez Díaz	49529438G	(drop indexes 558)
- Sara Serra 90585882A (replace others with NaN FIRST = OK!!) (drop indexes 25, 913)

KEEP ROW LOWEST employee_id (keep original row):
- Oriol	Ferrer	52590001R (drop index 1004)
- Ida	Karlsen Eriksen	55283701 (drop index 40)
- Claudia	Gil Ferrer	55297384F (drop index 1003)
- Alba	Muñoz	97954019T (drop index 1002 )  

Again, I re-ran the list of duplicate DNIs to double-check correct index row numbers prior to deletion.

In [36]:
# re-running list of duplicate DNIs so get corrected INDEX numbers for each (after previous fixes carried out)
duplicate_rows = df[df.duplicated(subset=['DNI'], keep=False)]
print(f"{len(duplicate_rows)} rows are duplictae DNIs \n")

display(duplicate_rows.sort_values('DNI'))

21 rows are duplictae DNIs 



,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
250,250,Laia,Navarro Vila,15909601K,Espanya,Bilbao,30.00,5.00,1958.00,D,1235 euros,True,True,Grup D,Analista junior,0.00,False,NaN,NaN,10.30
604,604,Laia,Navarro,15909601K,Espanya,Bilbao,30.00,5.00,1958.00,D,"1,676 �,�",True,False,Grup A,Analista,1.00,True,50.00,NaN,14.20
558,558,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.00,10.00,1981.00,Dona,"1,176 �,�",NaN,True,Grup A,Data Analyst,0.00,False,NaN,NaN,38.00
692,692,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.00,10.00,1981.00,Dona,"1663�,�",NaN,True,Grup C,Cap de projecte,NaN,True,80000.00,10.70,16.90
757,757,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.00,7.00,1973.00,D,"786�,�",NaN,True,Grup A,Analista,0.00,True,81381.00,6.50,13.90
1004,1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.00,7.00,1973.00,D,"786�,�",NaN,True,Grup A,Analista,0.00,True,81381.00,6.50,13.90
36,464,Ida,Karlsen Eriksen,55283701C,Noruega,Oslo,2.00,9.00,1992.00,D,"1,614 �,�",True,NaN,Grup B,Analista,3.00,NaN,NaN,NaN,10.60
40,1001,Ida,Karlsen Eriksen,55283701C,Noruega,Oslo,2.00,9.00,1992.00,D,"1,614 �,�",True,NaN,Grup B,Analista,3.00,NaN,NaN,NaN,10.60
730,730,Claudia,Gil Ferrer,55297384F,Espanya,València,1.00,11.00,1981.00,H,"1.789 �,�",NaN,True,Grup B,Responsable de vendes,0.00,True,29952.00,9.20,20.30
1003,1003,Claudia,Gil Ferrer,55297384F,Espanya,València,1.00,11.00,1981.00,H,"1.789 �,�",NaN,True,Grup B,Responsable de vendes,0.00,True,29952.00,9.20,20.30


In [37]:
# Laia Navarro Vila (adding 2nd surname from row to be deleted to the row that I am keeping):
df.loc[df['employee_id'] == 604, 'cognoms'] = ('Navarro Vila')


In [38]:
# deleting selected rows in df  (dropping final unwanted dupe DNI rows after changes needed made):

rows_to_drop = [250, 558, 25, 913, 1004, 40, 1003, 1002]

# dropping those listed:
df = df.drop(index=rows_to_drop)


In [39]:
# checking remaining dupe DNIs  = OK. left with just the 6 'NaN' DNIs
dupes_after = df[df.duplicated(subset=['DNI'], keep=False)]
print(f"{len(dupes_after)} rows are duplicate DNIs (AFTER) \n")
display(dupes_after.sort_values('DNI'))



6 rows are duplicate DNIs (AFTER) 



,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
89,89,Guillem,Pujol Costa,NaN,Espanya,A Coruña,17.00,7.00,1983.00,D,"1.574 �,�",True,NaN,Grup B,Project lead,4.00,False,NaN,NaN,14.60
104,104,Oriol,Vila Casals,NaN,Espanya,Madrid,30.00,9.00,1964.00,H,1987 euros,NaN,True,Grup B,Responsable de vendes,0.00,False,NaN,NaN,13.50
114,114,Pol,Serra,NaN,Espanya,Girona,4.00,12.00,1960.00,H,"1,178 �,�",NaN,True,Grup A,Data Analyst,NaN,False,NaN,NaN,16.20
142,142,Marta,Ruiz,NaN,Espanya,Palma,24.00,5.00,1982.00,H,"1,276 �,�",True,True,Grup A,Analista,0.00,False,NaN,NaN,15.70
281,281,Xavier,Vidal,NaN,Espanya,Alacant,2.00,7.00,1958.00,H,1179 euros,NaN,NaN,Grup A,Tècnic IT,0.00,True,63574.00,6.00,16.80
1005,1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### fixing rows without valid employee names in 'nom' (both are empty or dummy data)

After checking rows that have "???" or NaN in 'nom':  
- row 1006 can be deleted, dummy data (and also solves some extrem max/min values noticed)
- row 1005 has NaN for all values except employee_id

In [40]:

print(f"??? for nom: >>>>>> {df.loc[df['nom'] == '???']}","\n\n")


print(f" NaN in nom: >>>>>> {df.loc[df['nom'].isna()]}")

??? for nom: >>>>>>       employee_id  nom cognoms   DNI pais_d'origen       ciutat  \
1006         1006  ???     ###  XXXX        Narnia  Desconeguda   

      dia_de_naixement  mes_de_naixement  any_de_naixement genere  \
1006            -99.00             99.00           3000.00     ??   

     salari_mensual fills no_fills grup_professional càrrec  nombre_fills  \
1006          molts   NaN      NaN                 Z    ???           NaN   

     te_cotxe  km_anuals  consum_mitja_l_100km  temperatura_mitjana_ciutat  
1006      NaN        NaN                   NaN                         NaN   


 NaN in nom: >>>>>>       employee_id  nom cognoms  DNI pais_d'origen ciutat  dia_de_naixement  \
1005         1005  NaN     NaN  NaN           NaN    NaN               NaN   

      mes_de_naixement  any_de_naixement genere salari_mensual fills no_fills  \
1005               NaN               NaN    NaN            NaN   NaN      NaN   

     grup_professional càrrec  nombre_fills te_cotxe  

In [41]:
# dropping above identified rows
df =  df.drop([1005, 1006], axis=0)


In [42]:
# checking last rows in table to see ok after deletions:
df.tail(5)

,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
997,997,Jordi,Hernández,67755039Y,Espanya,Zaragoza,12.00,10.00,1961.00,Dona,"1,074 �,�",NaN,NaN,Grup A,Data Analyst,1.00,True,80000.00,12.00,12.00
998,998,Chloé,Dubois,66354268T,França,Nice,2.00,12.00,1958.00,H,"1,954 �,�",True,NaN,Grup B,Data Analyst,1.00,NaN,NaN,NaN,11.70
999,999,Adrià,Vila Díaz,57511543T,Espanya,Palma,31.00,12.00,1991.00,H,"2,010 �,�",True,NaN,Grup C,Analista,1.00,False,NaN,NaN,-10.00
1000,1000,Sara,Serra,90585882A,Espanya,Tarragona,28.00,5.00,2002.00,D,"1,073 �,�",NaN,True,Grup A,Administratiu,0.00,False,NaN,NaN,15.80
1001,651,Kasper,Pettersen,62042563B,Noruega,Oslo,11.00,5.00,1996.00,D,"2.290 �,�",NaN,True,Grup B,Senior analyst,0.00,True,80000.00,0.50,6.50


### DNI fixes COMPLETE (final inspection and list for HR)

After all noted DNI fixes executed, I run a final check on duplicated values in this column and generate the list, for the HR team, of employees to be contacted to retrieve the missing data.

In [43]:
# LIST FOR HR TEAM >> EMPLOYEES TO CONTACT TO CONFIRM CORRECT DNI FOR FILES:

display(df.loc[df['DNI'].isna(), ['employee_id', 'nom', 'cognoms', 'DNI', 'ciutat', 'càrrec']])


,employee_id,nom,cognoms,DNI,ciutat,càrrec
89,89,Guillem,Pujol Costa,NaN,A Coruña,Project lead
104,104,Oriol,Vila Casals,NaN,Madrid,Responsable de vendes
114,114,Pol,Serra,NaN,Girona,Data Analyst
142,142,Marta,Ruiz,NaN,Palma,Analista
281,281,Xavier,Vidal,NaN,Alacant,Tècnic IT


#### ... duplicate employee? Adrià Casals (= NO)

In my final inspection double check the data for two employees with the same name.  
As most of the data is complete for each and there are no duplicate values, I decide these are both valid employees and that just coincidentally have the same name. No action required

In [44]:
display(df.loc[(df['cognoms'] == 'Casals') & (df['nom'] == 'Adrià')])

,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
318,318,Adrià,Casals,89593852X,Espanya,Valladolid,6.00,5.00,1966.00,Home,"2,675 �,�",True,NaN,Grup C,Cap de projecte,1.00,True,30000.00,12.70,8.90
685,685,Adrià,Casals,38259178F,Espanya,Zaragoza,11.00,11.00,1977.00,H,"1,923 �,�",NaN,True,Grup C,Responsable de vendes,0.00,False,NaN,NaN,15.50


# **NIVELL 1 - EXERCISE 3**  
---


#### **Transformacions necessàries i preparació del dataset**

En aquest exercici hauràs d'aplicar transformacions bàsiques però essencials, orientades a millorar la qualitat de les dades i facilitar l'anàlisi posterior.

Aspectes a revisar i transformar:

- » Els valors categòrics estan normalitzats i són consistents?
- » Els valors nuls són coherents amb la informació disponible?
- » Hi ha redundància de variables que pugui generar problemes analítics?
- » Les variables contextuals (com temperatura, salari) estan completes i preparades per a l'anàlisi?  

Concretament, hauràs de:

- » Normalitzar els valors categòrics per evitar ambigüitats (per exemple, valors que representen el mateix concepte però estan escrits de manera diferent).
- » Revisar la coherència dels valors nuls, assegurant que:
Les persones que tenen quilometratge anual i consum mitjà associats disposen de cotxe. En els casos en què consti consum mitjà però no quilometratge anual, caldrà decidir com resoldre aquesta incoherència de manera justificada.
Les persones que tenen un valor en el nombre de fills efectivament tenen fills.
- » Unificar la informació sobre fills en una única columna, ja que mantenir diverses columnes amb informació complementària no és òptim i pot generar problemes analítics (redundància o multicolinealitat).
- » Validar la temperatura mitjana. En cas que aquesta informació falti, hauràs d'imputar-la utilitzant un criteri raonable (per ciutat o per un nivell superior justificat).
- » Revisar i imputar els salaris faltants, aplicant criteris analítics coherents (com ara càrrec, grup professional, ciutat, gènere o franja d'edat).


---



#### **Outline of my process for the transformations phase of the EDA project:**  


Given the complexity of the dataset, and the relationships between different variables that should be assessed for coherence, or how best to treat missing values, I have opted to systematically inspect the data column by column to reach a decision in each case of the optimum actions to be taken.

When different columns are clearly related (for example the various columns on vehicle statistics), I have assesed these in groups to assure coherence between the related data points.   

All code executed is organised in sections for each segment below.

---



Following is a **summary of the actions** taken per column / group of related columns:  

- **'càrrec'**
    - standardization of job title values
- **'salari_mensual'**  
    - convert the salaries written in letters and not numbers
    - correct datatype and format (standardize decimal representation)
    - evaluate imputation of missing values
    - imputate missing values
- **'genere'**  
    - establish accepted values to be validated
    - fill missing values
- **Child related columns**
    - ['fills', 'no_fills', 'nombre_fills']
    - correct datatypes
    - validate coherence and null values in 'fills', 'no_fills'
    - combine the information so just 1 column remains de denote children
    - remove now redundant columns ('fills' and 'no_fills')
- **Date of Birth related columns**
    - ['dia_de_naixement', 'mes_de_naixement', 'any_de_naixement']
    - identify and evaluate impossible calendar values
    - evaluate implications of missing values
    - combine and convert data into single new column created, 'DOB' (date format)
    - remove now redundant columns (3 original columns)
- **'temperatura_mitja_ciutat'**
    - evaluate coherence of values
    - standardize values per city
- **Vehicle related columns**
    - ['te_cotxe', 'km_anuals', 'consum_mitja_l_100km']
    - check coherence between the 3 columns and validate data
    - correct datatypes
    - identify extreme values (outliers) and negative values (to NaN)
    - evaluate imputation options
    - imputate missing values


## CARREC FIX

- **'càrrec'**
    - standardization of job title values

In [45]:
# checking current titles in 'carrec':

jobs_list = []
for group in df.groupby(['càrrec']).groups:
  jobs_list.append(group)
print("list of jobs:")
print(jobs_list)

list of jobs:
['Admin.', 'Administratiu', 'Analista', 'Analista junior', 'Analyst', 'Cap de Projecte', 'Cap de projecte', 'Data Analyst', 'Data analyst', 'Project lead', 'Responsable de vendes', 'Senior analyst', 'Tècnic IT']


Have inspected the many values for "analyst" related positions, some including qualifiers for level of seniority, and have determined the following...

ASSUMPTIONS:
- Analyst (general) and Data Analyst are different specialisations.
- Will maintain the 3 levels of seniority, as no indicators suggest should be reduced to just junior and senior. Therefore, those that are not labelled already Jr or Sr are neither:
    - Junior, Senior or un_defined

In [46]:
# CHECKING IF WAY TO TELL SENIOR OR JUNIOR DATA ANALYST: = NO.
data_analyst = ['Analista', 'Analista junior', 'Analyst', 'Data Analyst', 'Data analyst', 'Senior analyst']

# need to see all rows:
pd.set_option('display.max_rows', None)
display(df.loc[df['càrrec'].isin(data_analyst), ['càrrec', 'grup_professional', 'salari_mensual']].sort_values(['càrrec', 'grup_professional']))
pd.set_option('display.max_rows', 10)

,càrrec,grup_professional,salari_mensual
62,Analista,Grup A,"1251�,�"
80,Analista,Grup A,"827 �,�"
142,Analista,Grup A,"1,276 �,�"
147,Analista,Grup A,"975 �,�"
154,Analista,Grup A,"1298�,�"
170,Analista,Grup A,"1,156 �,�"
188,Analista,Grup A,"1,026 �,�"
204,Analista,Grup A,"987 �,�"
211,Analista,Grup A,1132 euros
265,Analista,Grup A,"1.026 �,�"


Based on the list of current values in 'càrrec' printed above, I construct a dictionary with the standard values defined to clean and normalize the data.

In [47]:
# creating a dictionary with the corrections (using as basis printed jobs list):
jobs_dict = {'Administratiu' : ['Administratiu', 'Admin.', 'Administriu'],
            'Analista' : ['Analista', 'Analyst'],
            'Analista Junior' : ['Analista junior'],
            'Cap de Projecte' : ['Cap de Projecte', 'Cap de projecte'],
            'Data Analyst' : ['Data Analyst', 'Data analyst'],
            'Project Lead' : ['Project lead'],
            'Responsable de Vendes' : ['Responsable de vendes'],
            'Analista Senior' : ['Senior analyst'],
            'Tècnic IT' : ['Tècnic IT']}

# flipping keys with values, so each value in the lists becomes a single key:
jobs_clean = {item:k for k,value_list in jobs_dict.items() for item in value_list}


In [48]:
# checking dictionary created
print("\ncleaned and flipped jobs dictionary to apply:")
print(jobs_clean)


cleaned and flipped jobs dictionary to apply:
{'Administratiu': 'Administratiu', 'Admin.': 'Administratiu', 'Administriu': 'Administratiu', 'Analista': 'Analista', 'Analyst': 'Analista', 'Analista junior': 'Analista Junior', 'Cap de Projecte': 'Cap de Projecte', 'Cap de projecte': 'Cap de Projecte', 'Data Analyst': 'Data Analyst', 'Data analyst': 'Data Analyst', 'Project lead': 'Project Lead', 'Responsable de vendes': 'Responsable de Vendes', 'Senior analyst': 'Analista Senior', 'Tècnic IT': 'Tècnic IT'}


I use replace() with map() to assign the standardized values for 'cárrec' to each row

In [49]:
# printing current values before applying the fix:
print(f"\ncurrent job titles (BEFORE FIX)\n")
for title in (df['càrrec'].unique().tolist()):
  print(title)


current job titles (BEFORE FIX)

Cap de projecte
Senior analyst
Tècnic IT
Data Analyst
Administratiu
Responsable de vendes
Analista junior
Project lead
Admin.
Analyst
Analista
Data analyst
Cap de Projecte


In [50]:
# replacing values in df using jobs_clean dictionary and .map():
df['càrrec'] = df['càrrec'].map(jobs_clean)

In [51]:
# checking fix ok:  = OK

df.groupby('càrrec')['employee_id'].count()

,employee_id
càrrec,
Administratiu,109
Analista,123
Analista Junior,106
Analista Senior,122
Cap de Projecte,107
Data Analyst,157
Project Lead,18
Responsable de Vendes,123
Tècnic IT,129


## SALARY FIX

- **'salari_mensual'**  
    - convert the salaries written in letters and not numbers
    - correct datatype and format (standardize decimal representation)
    - evaluate imputation of missing values
    - imputate missing values

### fixing salaries written in words and not numbers

In [52]:
# checking text values in salary column:
df.sort_values('salari_mensual', ascending = False)

,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
749,749,Hugo,Robert,91224076E,França,Paris,31.00,1.00,1979.00,D,tres mil,NaN,True,Grup A,Analista Senior,NaN,False,NaN,NaN,14.00
994,994,Adrià,Hernández Ramos,80771894Q,Espanya,Bilbao,24.00,6.00,1990.00,A,tres mil,NaN,True,Grup A,Analista Senior,NaN,True,80000.00,25.00,18.20
339,339,Sergio,Álvarez,14978636K,Espanya,Màlaga,25.00,7.00,1961.00,D,tres mil,True,NaN,Grup A,Data Analyst,1.00,NaN,NaN,NaN,18.10
567,567,Alba,Ferrer Díaz,64234848R,Espanya,Màlaga,1.00,7.00,1983.00,D,tres mil,True,NaN,Grup A,Tècnic IT,2.00,NaN,NaN,NaN,14.80
428,428,Aina,Puig Vila,38182016V,Espanya,Alacant,27.00,4.00,1999.00,Dona,tres mil,True,NaN,Grup A,Analista Junior,1.00,True,80000.00,5.80,16.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714,714,Oriol,Álvarez,34131353V,Espanya,Las Palmas de Gran Canaria,17.00,1.00,1966.00,H,"1,005 �,�",NaN,True,Grup A,Analista Senior,0.00,NaN,NaN,NaN,22.30
306,306,Sergio,Martínez Álvarez,93906615X,Espanya,Bilbao,25.00,1.00,1995.00,Dona,"1,003 �,�",NaN,True,Grup A,Tècnic IT,0.00,False,NaN,NaN,NaN
13,13,Emilia,Hoffmann,66851377P,Alemanya,Stuttgart,21.00,2.00,1961.00,D,"1,003 �,�",NaN,True,Grup A,Project Lead,NaN,True,6931.00,12.20,12.40
988,988,David,Roca Pérez,31701972H,Espanya,Girona,3.00,5.00,1975.00,H,"1,002 �,�",True,NaN,Grup A,Analista,1.00,True,50453.00,11.80,9.50


In [53]:
# eg salary values written as words... :
df.loc[df['salari_mensual'] == 'mil vuit-cents']['salari_mensual'].head()

,salari_mensual
14,mil vuit-cents
109,mil vuit-cents
166,mil vuit-cents
206,mil vuit-cents
359,mil vuit-cents


Construct a dictionary with the following identified values in text to be replaced with digits or correct NaN value: "mil vuit-cents	", "dos mil	", "n/d	" and 'tres mil'

In [54]:
# creating dictionary for mapping corrections
dict_salari = {'mil vuit-cents': 1800.00, 'dos mil': 2000.00, 'n/d': np.nan, 'tres mil': 3000.00}

applying the dictionary to the salary column using .replace()

In [55]:
df['salari_mensual'] = df['salari_mensual'].replace(dict_salari)

In [56]:
# checking corrected:
corrected_words = ["mil vuit-cents	", "dos mil	", "n/d	" and 'tres mil']
print(f"words still present: {df.loc[df['salari_mensual'].isin(corrected_words), 'salari_mensual'].count()}")

words still present: 0


In [57]:
# checking NaN correctly assigned and no other data lost = ok, get 14, which is correct (original missing salary data)
df.loc[df['salari_mensual'].isna()]

,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,salari_mensual,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat
93,93,Pau,Gil,48532662J,Espanya,Girona,0.00,13.00,2026.00,Dona,NaN,True,NaN,Grup A,Data Analyst,4.00,False,NaN,NaN,14.50
146,146,Luca,Gallo Ferrari,74468586Y,Itàlia,Torino,20.00,11.00,2000.00,D,NaN,NaN,True,Grup A,Analista Junior,NaN,False,NaN,NaN,14.40
159,159,Anna,Fernández,65446317R,Espanya,Palma,10.00,9.00,1982.00,D,NaN,True,NaN,Grup A,Cap de Projecte,2.00,False,NaN,NaN,NaN
254,254,Zoé,Thomas Dubois,89260394J,França,Marseille,13.00,8.00,1992.00,H,NaN,True,NaN,Grup A,Analista Senior,2.00,NaN,NaN,NaN,20.60
291,291,Thomas,Lefebvre Moreau,85300576W,França,Lille,28.00,8.00,1977.00,NC,NaN,True,NaN,Grup C,Analista Senior,4.00,False,NaN,NaN,12.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
875,875,Laia,Vidal Romero,39330100H,Espanya,Sevilla,24.00,8.00,2004.00,D,NaN,True,NaN,Grup A,Cap de Projecte,2.00,False,NaN,NaN,20.30
917,917,Claudia,Gil Fernández,19213734E,Espanya,Palma,3.00,7.00,1971.00,M,NaN,NaN,True,Grup C,Analista Senior,NaN,True,81219.00,6.60,13.30
953,953,Luca,Rizzo Ferrari,28245984Z,Itàlia,Torino,29.00,6.00,2002.00,NaN,NaN,True,NaN,Grup A,Analista Senior,2.00,False,NaN,NaN,13.60
955,955,Jordi,Martínez Gil,13976162W,Espanya,Zaragoza,3.00,8.00,1967.00,D,NaN,True,NaN,Grup A,Cap de Projecte,1.00,NaN,NaN,NaN,-5.00


### fixing salary data type and format

In [58]:
# checking original values:

listed_b4= df['salari_mensual'].to_list()
print("before\n", listed_b4)

before
 ['1,469 �,�', '2,718 �,�', '1358 euros', '1,478 �,�', '1,284 �,�', '2,069 �,�', '2,349 �,�', '2,855 �,�', '1,338 �,�', '1,450 �,�', '982 �,�', '2,419 �,�', '3,672 �,�', '1,003 �,�', 1800.0, '1,336 �,�', '982 �,�', '1,385 �,�', '1,751 �,�', '889 euros', '1121 euros', '1,880 �,�', '1,094 �,�', '1479�,�', '1,840 �,�', '2,660 �,�', '751 �,�', '2,628 �,�', '2400 euros', '2564�,�', '927�,�', '2,510 �,�', '1,538 �,�', '3275 euros', '1,974 �,�', '1,614 �,�', '1,014 �,�', '2132�,�', 2000.0, '1,967 �,�', '1438�,�', '1,923 �,�', '1,090 �,�', '1,032 �,�', '2130�,�', '1939 euros', '1739�,�', '1,112 �,�', '1,714 �,�', '1,773 �,�', '1647 euros', '1,866 �,�', '1,799 �,�', '1,413 �,�', '1.021 �,�', '1156�,�', '1597�,�', '1,242 �,�', '2,125 �,�', '3016�,�', '1251�,�', '1,681 �,�', '2,418 �,�', '1664 euros', '1702�,�', '1672�,�', '2.275 �,�', '1.798 �,�', '1394�,�', '1,013 �,�', '1000 euros', '1,119 �,�', '956 �,�', '1,156 �,�', '1,058 �,�', '2,958 �,�', '1092 euros', '3,063 �,�', '827 �,�', '1,2

In order to make sure all the many format variations are fixed without losing or corrupting any of the data, i first assess the different format patterns present

In [59]:
# making copy for preliminary actions for salary datatype changes:
df_salary = df.copy()


In [60]:
# in the copy df, adding a column to show the various formats used in the salary column:

# Replace all digits with '#' to reveal the pattern structure
df_salary["pattern"] = df_salary["salari_mensual"].str.replace(r"\d", "#", regex=True)

# Use value_counts() to list the different formats and number of times they appear:
format_list = df_salary["pattern"].value_counts()

print(format_list)

pattern
#,### �,�     494
####�,�       136
### �,�       111
#### euros    107
#.### �,�      58
###�,�         34
### euros      17
#.###           1
Name: count, dtype: int64


From the above patterns listed, I can now determine all of the changes needed to clean the characters and prepare the values ready for converting the dataype

In [61]:
# printing values in main df before fix, to check works ok...
df_listed_b4= df['salari_mensual'].to_list()

print("before in df\n", df_listed_b4)

before in df
 ['1,469 �,�', '2,718 �,�', '1358 euros', '1,478 �,�', '1,284 �,�', '2,069 �,�', '2,349 �,�', '2,855 �,�', '1,338 �,�', '1,450 �,�', '982 �,�', '2,419 �,�', '3,672 �,�', '1,003 �,�', 1800.0, '1,336 �,�', '982 �,�', '1,385 �,�', '1,751 �,�', '889 euros', '1121 euros', '1,880 �,�', '1,094 �,�', '1479�,�', '1,840 �,�', '2,660 �,�', '751 �,�', '2,628 �,�', '2400 euros', '2564�,�', '927�,�', '2,510 �,�', '1,538 �,�', '3275 euros', '1,974 �,�', '1,614 �,�', '1,014 �,�', '2132�,�', 2000.0, '1,967 �,�', '1438�,�', '1,923 �,�', '1,090 �,�', '1,032 �,�', '2130�,�', '1939 euros', '1739�,�', '1,112 �,�', '1,714 �,�', '1,773 �,�', '1647 euros', '1,866 �,�', '1,799 �,�', '1,413 �,�', '1.021 �,�', '1156�,�', '1597�,�', '1,242 �,�', '2,125 �,�', '3016�,�', '1251�,�', '1,681 �,�', '2,418 �,�', '1664 euros', '1702�,�', '1672�,�', '2.275 �,�', '1.798 �,�', '1394�,�', '1,013 �,�', '1000 euros', '1,119 �,�', '956 �,�', '1,156 �,�', '1,058 �,�', '2,958 �,�', '1092 euros', '3,063 �,�', '827 �,�'

In [62]:
# convert to string and strip surrounding whitespace
df['salari_mensual'] = df['salari_mensual'].astype(str).str.strip()

# strip currency symbols and euro text
df['salari_mensual'] = (df['salari_mensual']
                                      .str.replace('�,�', '', regex=False)
                                      .str.replace('euros', '', regex=False)
                                      .str.replace('euro', '', regex=False)
                                      .str.replace('€', '', regex=False)
                                      .str.strip())

# remove thousands commas
df['salari_mensual'] = df['salari_mensual'].str.replace(r',(?=\d{3}(?:[^\d]|$))', '', regex=True)

# remove thousands dots
df['salari_mensual'] = df['salari_mensual'].str.replace(r'\.(?=\d{3}(?:[^\d]|$))', '', regex=True)

# convert remaining decimal commas to dots
df['salari_mensual'] = df['salari_mensual'].str.replace(',', '.', regex=False)

# clean any non-numeric characters remaining
df['salari_mensual'] = df['salari_mensual'].str.replace(r'[^\d.]', '', regex=True)

# convert empty or nan strings to real NaN (as lost in string conversions above)
df['salari_mensual'] = df['salari_mensual'].replace({'': np.nan, 'nan': np.nan, 'None': np.nan})

# convert to float
df['salari_mensual'] = df['salari_mensual'].astype(float)


In [63]:
# printing values after fix to df, to check worked ok...

df_listed_after= df['salari_mensual'].to_list()
print("after fix in df\n", df_listed_after)

after fix in df
 [1469.0, 2718.0, 1358.0, 1478.0, 1284.0, 2069.0, 2349.0, 2855.0, 1338.0, 1450.0, 982.0, 2419.0, 3672.0, 1003.0, 1800.0, 1336.0, 982.0, 1385.0, 1751.0, 889.0, 1121.0, 1880.0, 1094.0, 1479.0, 1840.0, 2660.0, 751.0, 2628.0, 2400.0, 2564.0, 927.0, 2510.0, 1538.0, 3275.0, 1974.0, 1614.0, 1014.0, 2132.0, 2000.0, 1967.0, 1438.0, 1923.0, 1090.0, 1032.0, 2130.0, 1939.0, 1739.0, 1112.0, 1714.0, 1773.0, 1647.0, 1866.0, 1799.0, 1413.0, 1021.0, 1156.0, 1597.0, 1242.0, 2125.0, 3016.0, 1251.0, 1681.0, 2418.0, 1664.0, 1702.0, 1672.0, 2275.0, 1798.0, 1394.0, 1013.0, 1000.0, 1119.0, 956.0, 1156.0, 1058.0, 2958.0, 1092.0, 3063.0, 827.0, 1201.0, 1133.0, 878.0, 1163.0, 1008.0, 1436.0, 1263.0, 1890.0, 1574.0, 1164.0, 2257.0, 1086.0, nan, 1455.0, 2488.0, 1700.0, 2166.0, 1392.0, 2359.0, 918.0, 1075.0, 2627.0, 1100.0, 1987.0, 2138.0, 1268.0, 1957.0, 1549.0, 1800.0, 1825.0, 1438.0, 993.0, 1240.0, 1178.0, 1244.0, 810.0, 2901.0, 932.0, 1082.0, 1567.0, 2180.0, 1058.0, 2303.0, 2111.0, 1019.0, 2227.

In [64]:
# check describe() to see ranges and confirm decimal point correctly positioned after fix

df['salari_mensual'].describe()

,salari_mensual
count,980.00
mean,1566.02
std,622.49
min,673.00
25%,1086.75
50%,1410.50
75%,1896.00
max,3672.00


In [65]:
# checking final wages formats in df: OK

final_wages_listed = df['salari_mensual'].dropna().unique().tolist()
final_wages_listed.sort()
for wage in final_wages_listed:
  print(wage)


673.0
701.0
718.0
722.0
723.0
724.0
726.0
743.0
746.0
747.0
751.0
759.0
765.0
766.0
770.0
771.0
774.0
775.0
778.0
779.0
781.0
783.0
785.0
786.0
792.0
795.0
796.0
797.0
803.0
806.0
807.0
808.0
810.0
816.0
817.0
819.0
821.0
822.0
823.0
824.0
827.0
830.0
833.0
838.0
842.0
843.0
844.0
845.0
847.0
850.0
854.0
856.0
857.0
858.0
860.0
861.0
863.0
864.0
866.0
867.0
868.0
871.0
874.0
878.0
880.0
881.0
885.0
886.0
888.0
889.0
891.0
896.0
897.0
901.0
902.0
903.0
904.0
906.0
909.0
918.0
919.0
920.0
922.0
926.0
927.0
929.0
931.0
932.0
933.0
935.0
937.0
938.0
939.0
940.0
943.0
946.0
947.0
950.0
951.0
952.0
956.0
958.0
962.0
966.0
967.0
968.0
971.0
975.0
976.0
978.0
979.0
980.0
982.0
985.0
987.0
988.0
989.0
990.0
993.0
994.0
995.0
997.0
998.0
999.0
1000.0
1001.0
1002.0
1003.0
1005.0
1007.0
1008.0
1012.0
1013.0
1014.0
1015.0
1019.0
1020.0
1021.0
1023.0
1024.0
1025.0
1026.0
1027.0
1028.0
1029.0
1030.0
1031.0
1032.0
1034.0
1035.0
1041.0
1042.0
1043.0
1046.0
1047.0
1048.0
1054.0
1056.0
1057.0
1058.0
1060

In [66]:
# checking no data lost in conversions: = OK, still have just the 14 NaN values

missing_salaries = df.loc[df['salari_mensual'].isnull()]
display(missing_salaries[['employee_id', 'salari_mensual']])

,employee_id,salari_mensual
93,93,NaN
146,146,NaN
159,159,NaN
254,254,NaN
291,291,NaN
...,...,...
875,875,NaN
917,917,NaN
953,953,NaN
955,955,NaN


### evaluating imputation of missing salaries

Need to decide whether and how to impute missing values, by studying if any correlation with other variables in the dataset

In [67]:
missing_salary = pd.isnull(df['salari_mensual'])

print(f'There are {df[missing_salary]['employee_id'].count()} employees without salary data:')
display(df[missing_salary].sort_values('càrrec')[['employee_id', 'salari_mensual', 'càrrec', 'grup_professional']])


There are 14 employees without salary data:


,employee_id,salari_mensual,càrrec,grup_professional
307,307,NaN,Analista,Grup C
146,146,NaN,Analista Junior,Grup A
254,254,NaN,Analista Senior,Grup A
291,291,NaN,Analista Senior,Grup C
917,917,NaN,Analista Senior,Grup C
...,...,...,...,...
956,956,NaN,Cap de Projecte,Grup A
93,93,NaN,Data Analyst,Grup A
384,384,NaN,Data Analyst,Grup B
446,446,NaN,Tècnic IT,Grup A


Checking values to decide imputation for missing salaries by testing for correlation with any of the other variables in the dataset.  

**NB: In the tests I run to check for correlations, the largest variance bewteen min and max of their median is the better indication of a relationship**  
*(similar to an alternative hypothesis result in the ANOVA test, but here on median rather than mean, and both individual and grouped categories)*

**>> Testing for correlation with just 1-level grouping aggregations (the individual categorical variables)**

Overview of the influence of the different categorical columns on salary data


In [68]:
# categorical columns to check:
categ_columns = ["pais_d'origen", "ciutat", "grup_professional", "càrrec"]

print("->> all categorical columns (1-LEVEL AGGREGATIONS)>>>:")

# Loop through the list of columns to check how grouping by the different variables affects the median of salari_mensual:
for cat in categ_columns:
  stats_salari = df.groupby(cat)['salari_mensual'].agg(['count', 'median', 'mean', 'std', 'max', 'min'])

  # Calculate difference between max and min median
  max_med_salari = stats_salari['median'].max()
  min_med_salari = stats_salari['median'].min()
  diff_med_salari = max_med_salari - min_med_salari

  # print summary and stats for each tested:
  print(f"\n>>> Category: {cat} <<<")
  print(f"Max Median: {max_med_salari:,.1f} | Min Median: {min_med_salari:,.1f} | Difference: {diff_med_salari:,.1f}\n")

->> all categorical columns (1-LEVEL AGGREGATIONS)>>>:

>>> Category: pais_d'origen <<<
Max Median: 1,510.0 | Min Median: 1,348.0 | Difference: 162.0


>>> Category: ciutat <<<
Max Median: 1,810.0 | Min Median: 995.5 | Difference: 814.5


>>> Category: grup_professional <<<
Max Median: 3,130.5 | Min Median: 1,072.0 | Difference: 2,058.5


>>> Category: càrrec <<<
Max Median: 1,643.0 | Min Median: 1,219.0 | Difference: 424.0



**2-level grouping aggregations (combination of most influential variables in previous test)**

In [69]:
# >>>>>>>>>> check grouping by 2-levels:
print("->> pairs of categorical columns (2-LEVEL AGGREGATIONS)>>>:")


# >>>>>>>>>>> check grouping by both grup_professional and ciutat (2-level):

df_grup_ciutat = df.groupby(['grup_professional', 'ciutat']) ['salari_mensual'].agg(['count', 'max', 'min', 'mean', 'median', 'std'])

# Calculate difference between max and min median
max_med_grup_ciutat = df_grup_ciutat['median'].max()
min_med_grup_ciutat = df_grup_ciutat['median'].min()
diff_med_grup_ciutat = max_med_grup_ciutat - min_med_grup_ciutat

# print summary and full grup_ciutat stats:
print(f"\n>>> Grouping by: grup_professional and ciutat (highest individual variances) <<<")
print(f"Max Median: {max_med_grup_ciutat:,.1f} | Min Median: {min_med_grup_ciutat:,.1f} | Difference: {diff_med_grup_ciutat:,.1f}")


# >>>>>>>>>>> check grouping by both càrrec and grup_professional (2-level):

df_carrec_grup = df.groupby(['càrrec', 'grup_professional']) ['salari_mensual'].agg(['count', 'max', 'min', 'mean', 'median', 'std'])

# Calculate difference between max and min median
max_med_carrec_grup = df_carrec_grup['median'].max()
min_med_carrec_grup = df_carrec_grup['median'].min()
diff_med_carrec_grup = max_med_carrec_grup - min_med_carrec_grup

# print summary and full carrec_grup stats:
print(f"\n>>> Grouping by: càrrec and grup_professional <<<")
print(f"Max Median: {max_med_carrec_grup:,.1f} | Min Median: {min_med_carrec_grup:,.1f} | Difference: {diff_med_carrec_grup:,.1f}")


->> pairs of categorical columns (2-LEVEL AGGREGATIONS)>>>:

>>> Grouping by: grup_professional and ciutat (highest individual variances) <<<
Max Median: 3,610.0 | Min Median: 842.0 | Difference: 2,768.0

>>> Grouping by: càrrec and grup_professional <<<
Max Median: 3,582.0 | Min Median: 983.5 | Difference: 2,598.5


**Decision on imputing missing salary data:**  
As there is much higher variance between minimum and maximum median values for salari_mensual when the data is group by either grup_professional or ciutat (1-level aggregation).  

Combining both of these categorical aggregations (**ciutat and grup_professional**), to convert it into a 2-level grouping aggregation, gives the highest resulting variance between minimum and maximum median values.  

These results suggest that using the median values from that 2-level aggregation to impute missing salaries would give a more reliable indicator of salary than just the specific job positions or grup_professional, for example, on their own.

### imputating missing salaries in df

In [70]:
# select the rows with missing data:
df_salary_missing = df['salari_mensual'].isna()

# reference table with median for combination of grup and ciutat:
df_medians_2level = df.groupby(['grup_professional', 'ciutat']) ['salari_mensual'].agg(['median'])

# display(df_medians_2level)

# construct dictionary to use in map() when applying fix to missing salaries:
medians_salari= df_medians_2level.to_dict()

# checking dictionary created
print(medians_salari)

{'median': {('Grup A', 'A Coruña'): 979.0, ('Grup A', 'Alacant'): 1145.0, ('Grup A', 'Barcelona'): 1006.0, ('Grup A', 'Bergen'): 1074.0, ('Grup A', 'Berlin'): 1078.0, ('Grup A', 'Bilbao'): 1155.0, ('Grup A', 'Bologna'): 1003.0, ('Grup A', 'Bordeaux'): 1117.0, ('Grup A', 'Düsseldorf'): 1058.0, ('Grup A', 'Firenze'): 1258.0, ('Grup A', 'Genova'): 1121.5, ('Grup A', 'Girona'): 1142.0, ('Grup A', 'Hamburg'): 1128.5, ('Grup A', 'Köln'): 842.0, ('Grup A', 'Las Palmas de Gran Canaria'): 956.0, ('Grup A', 'Leipzig'): 1236.0, ('Grup A', 'Lille'): 1092.0, ('Grup A', 'Lyon'): 999.5, ('Grup A', 'Madrid'): 1011.0, ('Grup A', 'Marseille'): 975.0, ('Grup A', 'Milano'): 1056.5, ('Grup A', 'Murcia'): 1142.5, ('Grup A', 'Màlaga'): 1131.0, ('Grup A', 'München'): 933.5, ('Grup A', 'Nantes'): 884.0, ('Grup A', 'Napoli'): 874.0, ('Grup A', 'Nice'): 1129.5, ('Grup A', 'Oslo'): 978.0, ('Grup A', 'Palma'): 1075.0, ('Grup A', 'Paris'): 1270.5, ('Grup A', 'Roma'): 1027.0, ('Grup A', 'Sevilla'): 1087.5, ('Grup A'

In [71]:
# zipping grup and ciutat to make the lookup key for the nested dictionary
# saving combined_key zip action as a list, so always available:

combined_key = list(zip(df['grup_professional'], df['ciutat']))

In [72]:
# use zipped list of tuples (grup, ciutat) for applying the map() in the fillna operation:
df['salari_mensual_def'] = df['salari_mensual'].fillna(pd.Series(combined_key, index=df_salary.index).map(medians_salari['median']))

In [73]:
# checking fix worked and NaN salaries filled with combined categories' median values:

df.loc[df_salary_missing][['employee_id', 'grup_professional', 'ciutat', 'salari_mensual', 'salari_mensual_def']].sort_values(['grup_professional', 'ciutat'])

,employee_id,grup_professional,ciutat,salari_mensual,salari_mensual_def
860,860,Grup A,Bordeaux,NaN,1117.00
93,93,Grup A,Girona,NaN,1142.00
446,446,Grup A,Las Palmas de Gran Canaria,NaN,956.00
254,254,Grup A,Marseille,NaN,975.00
159,159,Grup A,Palma,NaN,1075.00
...,...,...,...,...,...
956,956,Grup A,Zaragoza,NaN,1012.00
384,384,Grup B,Barcelona,NaN,1585.00
307,307,Grup C,Barcelona,NaN,2600.00
291,291,Grup C,Lille,NaN,2065.50


In [74]:
# checking sample from above (grup c + barcelona = 2600) value imputated correctly: = OK

# filtering the key-value for the sample above that was imputated:
print(medians_salari['median'][('Grup C', 'Barcelona')])



2600.0


In [75]:
# checking no NaN values in newly created column with values imputated and that the values present are ok: = OK
print(f"Null values in new salary column: {df['salari_mensual_def'].isna().sum()}\n")

display(df['salari_mensual_def'].describe())

Null values in new salary column: 0



,salari_mensual_def
count,994.00
mean,1562.74
std,622.05
min,673.00
25%,1082.25
50%,1408.00
75%,1895.00
max,3672.00


now that the new column is created and populated with either original salary value or the calculated result if the original data was missing, I can drop the original salary column.

In [76]:
# Drop original 'salari_mensual' column as now could cause ambiguity...
df = df.drop(columns=['salari_mensual'])

In [77]:
# checking remaining columns in dataset:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 994 entries, 0 to 1001
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   employee_id                 994 non-null    int64  
 1   nom                         994 non-null    object 
 2   cognoms                     994 non-null    object 
 3   DNI                         989 non-null    object 
 4   pais_d'origen               994 non-null    object 
 5   ciutat                      994 non-null    object 
 6   dia_de_naixement            994 non-null    float64
 7   mes_de_naixement            994 non-null    float64
 8   any_de_naixement            994 non-null    float64
 9   genere                      955 non-null    object 
 10  fills                       415 non-null    object 
 11  no_fills                    578 non-null    object 
 12  grup_professional           994 non-null    object 
 13  càrrec                      994 non-nul

## GENERE FIX

- **'genere'**  
    - establish accepted values to be validated (protocol)
    - fill missing values
    - check data coherent for all rows

### detail of aspects to fix

Decision needed on values accepted (original has 9 unique values).  

NOTE: In a real life scenario, before defining the gender values to be accpeted in this dataset, or any modifications to original values, I would **consult with the company Legal department and any Employee Resource Group** dedicated to supporting and protecting LGBTQ+ employees, regarding accepted registration of gender information to make sure that the process followed is legally correct and is not vulnerating any basic rights of those individuals included in the records.  

After inspection have initially decided to use just **2 values, 'H' and 'D'**, ( representing Home and Dona, as the dataset is in catalan), and base corrections on common first-name genders.  
Any rows that do not adhere to the common first-name genders will be treated as incorrect data (user error on entry) and modified.


In [78]:
# checking values and distribution:
grouped_genere = df.groupby('genere')
generes = grouped_genere.agg({'employee_id': 'count'})
generes


,employee_id
genere,
A,24
D,320
Dona,118
F,107
H,279
Home,50
M,34
NC,23


In [79]:
# checking missing data:
df['genere'].isna().sum()

np.int64(39)

NOTE: gender defined as 'A' is mix of male and female.  
As I have not been provided with a key for the current values in the series, I will need to check and fix manually as with the 39 NaN values


In [80]:
df.loc[df['genere'] == 'A', ['nom', 'genere']]

,nom,genere
116,Sara,A
209,Víctor,A
259,Camille,A
304,Marina,A
307,David,A
...,...,...
768,Sergio,A
805,Claudia,A
887,Anna,A
947,Marc,A


### Completing missing gender data

Will use a dictionary and .map() to replace/fill incorrect or missing values.

In a preliminary step, **I run the following code only ONCE** to dynamically create a dictionary to classify each unique name.  

The code:  
- loops through list of names requiring gender allocation
- asks user to enter either 'H' or 'D' for each
- saves this information in a dictionary format
  
The **output of this code** is saved below, providing the **reference dictionary to apply** to the gender column in the dataset

In [81]:
# user interaction required to create a dictionary to classify each unique name
# directly reapply SAVED generated dict in future sessions...

''' >>>>>> COMMENTING OUT AS ALREADY GENERATED and have saved the dictionary in previous session!!! >>>>>>

# empty dictionary to store Name > Gender mappings I input:
manual_gender_dict = {}

# make sure 'nom' column is clean (no whitespaces) first:
# df_genere_test['nom'] = df_genere_test[df['nom']].strip()

# Filter rows where genre is NaN again
missing_mask = df_genere_test['genere'].isna()
print(missing_mask.value_counts(dropna=False)) # 86 included in the mask

# Loop through unique names that need classification
unique_missing_names = df[missing_mask]['nom'].unique()
print(unique_missing_names.size) # 45 unique names to classify (will go in dict)

# loop through the unique names to get input for each and add to dict
for name in unique_missing_names:
    user_input = input(f"Name: {name} -> Enter 'H' or 'D' (or Enter to skip): ").strip().upper()
    if user_input in ['H', 'D']:
        manual_gender_dict[name] = user_input

# show dict created at end (to copy and save):
manual_gender_dict

'''

' >>>>>> COMMENTING OUT AS ALREADY GENERATED and have saved the dictionary in previous session!!! >>>>>>\n\n# empty dictionary to store Name > Gender mappings I input:\nmanual_gender_dict = {}\n\n# make sure \'nom\' column is clean (no whitespaces) first:\n# df_genere_test[\'nom\'] = df_genere_test[df[\'nom\']].strip()\n\n# Filter rows where genre is NaN again\nmissing_mask = df_genere_test[\'genere\'].isna()\nprint(missing_mask.value_counts(dropna=False)) # 86 included in the mask\n\n# Loop through unique names that need classification\nunique_missing_names = df[missing_mask][\'nom\'].unique()\nprint(unique_missing_names.size) # 45 unique names to classify (will go in dict)\n\n# loop through the unique names to get input for each and add to dict\nfor name in unique_missing_names:\n    user_input = input(f"Name: {name} -> Enter \'H\' or \'D\' (or Enter to skip): ").strip().upper()\n    if user_input in [\'H\', \'D\']:\n        manual_gender_dict[name] = user_input\n\n# show dict create

Dynamically generated mapping dictionary saved as 'gender_dict'

In [82]:
# saved dictionary for applying in main df and in each new session:

gender_dict = {'Antoine': 'H',
 'Xavier': 'H',
 'Carlos': 'H',
 'Alexia': 'D',
 'Elena': 'D',
 'Noa': 'D',
 'Katharina': 'D',
 'Sara': 'D',
 'Lucas': 'H',
 'Marco': 'H',
 'Víctor': 'H',
 'Chiara': 'D',
 'Camille': 'D',
 'Thomas': 'H',
 'Simone': 'D',
 'Lucía': 'D',
 'Marina': 'D',
 'David': 'H',
 'Laia': 'D',
 'Núria': 'D',
 'Hugo': 'H',
 'Anna': 'D',
 'Pol': 'H',
 'Marta': 'D',
 'Gabriele': 'H',
 'Luca': 'H',
 'Alba': 'D',
 'Carmen': 'D',
 'Alejandro': 'H',
 'Marc': 'H',
 'Adrià': 'H',
 'Marie': 'D',
 'Aina': 'D',
 'Sergio': 'H',
 'Alessandro': 'H',
 'Pau': 'H',
 'Mia': 'D',
 'Claudia': 'D',
 'Riccardo': 'H',
 'Elias': 'H',
 'Irene': 'D',
 'Felix': 'H',
 'Nil': 'H',
 'Joana': 'D',
 'Ilaria': 'D'}

In [83]:
# filtered copy of those needing modification:
fixing = df.loc[~df['genere'].isin(['H', 'D'])]  # using the ~ (alt gr + 4) makes it NOT isin...
fixing.head()


,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat,salari_mensual_def
0,1007,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.00,3.00,1958.00,Dona,True,NaN,Grup B,Cap de Projecte,3.00,True,32108.00,25.00,10.10,1469.00
4,4,David,Vila,82070937P,Espanya,Bilbao,29.00,11.00,1965.00,Dona,NaN,True,Grup B,Administratiu,NaN,True,NaN,11.80,13.10,1284.00
10,10,Joana,Torres Díaz,82772208R,Espanya,Valladolid,24.00,10.00,1961.00,F,NaN,True,Grup A,Administratiu,0.00,True,14178.00,7.70,38.00,982.00
16,16,Sara,Hernández Costa,91363974Q,Espanya,València,14.00,6.00,2004.00,Home,NaN,True,Grup A,Analista Junior,0.00,NaN,NaN,NaN,15.80,982.00
18,18,Nil,Fernández Vidal,69345683K,Espanya,Bilbao,15.00,8.00,1978.00,Home,NaN,True,Grup B,Analista Senior,NaN,True,64444.00,0.50,16.00,1751.00


Unifying values and defining NaN prior to applying the dictionary:

In [84]:
# can now apply gender fixes to main df...

# initial fixes to unify all known genders to same 2 values (H or D)
genre_clean = {'A':np.nan, 'Dona': 'D', 'F': 'D', 'Home': 'H', 'M': 'H', 'NC': np.nan }
df['genere'] = df['genere'].replace(genre_clean)


applying the cleaned gender_dict to main df, using .map():

In [85]:
# and applying the cleaned gender_dict to main df, using .map():

df['genere'] = df['genere'].fillna(df['nom'].map(gender_dict))


In [86]:
# checking how many values after fixes to main df... = OK ('H' and 'D')
df['genere'].value_counts(dropna=False)

,count
genere,
D,591
H,403


### corrections and modifications to original gender data

In addition to the missing data, some rows have been identified with contradicting data if we are basing the data optimisation on common first-name genders.  
(eg employee 4, 'David' as 'Dona' in original excel)

A simple way to check the common first-name gender "rule" is to see what gender value is most frecuent for each first name.

In [87]:
# showing all rows (not truncated):
# pd.set_option('display.max_rows', None)


# checking how many genders for each name....
names_genders = df.groupby(["nom"])["genere"].describe()

# there are 102 different names
display(names_genders)

# and 75 of them have more than 1 gender noted
display(names_genders.loc[names_genders['unique'] > 1])


# return to truncated rows view:
#pd.set_option('display.max_rows', 10)

,count,unique,top,freq
nom,,,,
Adrià,34,2,D,18
Aina,29,2,D,20
Alba,27,2,D,18
Alejandro,15,2,D,11
Alessandro,8,2,D,5
...,...,...,...,...
Valentina,4,2,D,3
Víctor,25,2,D,13
Xavier,27,2,D,17


,count,unique,top,freq
nom,,,,
Adrià,34,2,D,18
Aina,29,2,D,20
Alba,27,2,D,18
Alejandro,15,2,D,11
Alessandro,8,2,D,5
...,...,...,...,...
Valentina,4,2,D,3
Víctor,25,2,D,13
Xavier,27,2,D,17


Of the 102 different first names in the dataset, 75 of them have more than 1 gender recorded, which we can deduce means either incorrect data or of ambiguous reliability.


Furthermore, from the above test results, it does not seem advisable to blindly modify the gender for these 75 names with the top (most frequent) value.  

The following names appear to have incorrect gender as their 'top' value:

---
**top is 'H' ('Home')** for Zoé, Astrid, Chloé, Claudia, Hannah, Irene, Mia

---

**top is 'D' ('Dona')** for Adrià, Alejandro, Alessandro, Antoine, Carlos, David, Davide, Federico, Felix, Hugo, Leon, Luca, Nicolas, Nil, Oriol, Pau, Paul, Riccardo, Sergio, Sigrid, Thomas, Víctor, Xavier

---


From the list above of discrepancies in top gender for each name, I use .isin() to first reapply gender to the names in the list, and then I can use the top value for all the rows.

### fixing top gender in main df

In [88]:
# apply the correction to identified names that have the top gender wrong in the main df:

should_be_D = ['Zoé', 'Astrid', 'Chloé', 'Claudia', 'Hannah', 'Irene', 'Mia']
should_be_H = ['Adrià', 'Alejandro', 'Alessandro', 'Antoine', 'Carlos', 'David', 'Davide', 'Federico', 'Felix', 'Hugo', 'Leon', 'Luca', 'Max', 'Nicolas', 'Nil', 'Oriol', 'Pau', 'Paul', 'Riccardo', 'Sergio', 'Sigrid', 'Thomas', 'Víctor', 'Xavier']

masking_fixes_D = df['nom'].str.strip().isin(should_be_D)
masking_fixes_H = df['nom'].str.strip().isin(should_be_H)

# stripping any possible whitespaces in 'nom' column
df['nom'] = df['nom'].str.strip()

df.loc[masking_fixes_D, 'genere'] = 'D'
df.loc[masking_fixes_H, 'genere'] = 'H'

# checking worked: = OK
df.loc[df['nom'].isin(should_be_H)].sort_values('nom')


,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,fills,no_fills,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat,salari_mensual_def
999,999,Adrià,Vila Díaz,57511543T,Espanya,Palma,31.00,12.00,1991.00,H,True,NaN,Grup C,Analista,1.00,False,NaN,NaN,-10.00,2010.00
836,836,Adrià,Domènech,82613651V,Espanya,Alacant,25.00,11.00,1958.00,H,NaN,True,Grup A,Tècnic IT,0.00,False,NaN,NaN,14.40,838.00
825,825,Adrià,Domènech,51722201V,Espanya,Madrid,0.00,13.00,1940.00,H,True,NaN,Grup A,Analista Junior,2.00,NaN,NaN,NaN,16.10,1024.00
817,817,Adrià,Gil,81019783K,Espanya,Las Palmas de Gran Canaria,16.00,2.00,1994.00,H,NaN,True,Grup A,Responsable de Vendes,NaN,False,NaN,NaN,21.80,838.00
732,732,Adrià,Pujol,45404362D,Espanya,Palma,17.00,4.00,1997.00,H,NaN,True,Grup B,Analista,0.00,True,107127.00,NaN,14.00,1572.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
728,728,Xavier,Romero,59379352L,Espanya,Girona,13.00,4.00,2001.00,H,NaN,True,Grup B,Administratiu,NaN,False,NaN,NaN,14.20,1165.00
434,434,Xavier,Hernández,29950518C,Espanya,Màlaga,28.00,1.00,1962.00,H,NaN,True,Grup A,Data Analyst,NaN,True,71500.00,14.00,19.80,1031.00
509,509,Xavier,Romero Álvarez,38277083J,Espanya,Murcia,2.00,3.00,1969.00,H,True,NaN,Grup C,Cap de Projecte,1.00,False,NaN,NaN,NaN,2952.00
821,821,Xavier,Serra,38356463G,Espanya,Sevilla,6.00,8.00,1961.00,H,NaN,True,Grup A,Cap de Projecte,NaN,False,NaN,NaN,20.90,1191.00


In [89]:
# now checking top values again for gender per name.... (tops should have been fixed!)

# checking how many genders for each name....
names_genders = df.groupby(["nom"])["genere"].describe()

# to display all rows (not truncated view)...
with pd.option_context('display.max_rows', None):
    display(names_genders)


,count,unique,top,freq
nom,,,,
Adrià,34,1,H,34
Aina,29,2,D,20
Alba,27,2,D,18
Alejandro,15,1,H,15
Alessandro,8,1,H,8
Alessia,3,1,D,3
Alexia,25,2,D,17
Anna,26,2,D,19
Antoine,5,1,H,5


With top genders for each name now corrected, can do a replace of all values with their top gender per name

In [90]:
# List of 'top' gender for each name (making sure 'name' as index, to later do the map()) using the 'describe' results saved above
gender_map = names_genders['top']
gender_map.index = gender_map.index.astype(str).str.strip()

#create table with map() to fill the corrected values
correct_genders = df['nom'].str.strip().map(gender_map)

# recreate mask (from df, not from the above describe as before)
names_with_conflicts = df.groupby('nom')['genere'].transform('nunique') > 1

# 4. Apply to masked rows with .loc()
df.loc[names_with_conflicts, 'genere'] = correct_genders


In [91]:
#checking if worked: OK!
gender_grouped_df = df.groupby(["nom"])["genere"].describe()
# to display all rows TEMPORARILY (not truncated view)...
with pd.option_context('display.max_rows', None):
    display(gender_grouped_df)

,count,unique,top,freq
nom,,,,
Adrià,34,1,H,34
Aina,29,1,D,29
Alba,27,1,D,27
Alejandro,15,1,H,15
Alessandro,8,1,H,8
Alessia,3,1,D,3
Alexia,25,1,D,25
Anna,26,1,D,26
Antoine,5,1,H,5


In [92]:
# checking remaining null values in columns...

with pd.option_context('display.max_rows', None):
  display(df.isna().sum())


,0
employee_id,0
nom,0
cognoms,0
DNI,5
pais_d'origen,0
ciutat,0
dia_de_naixement,0
mes_de_naixement,0
any_de_naixement,0
genere,0


## FILLS / NO_FILLS FIX

- **Child related columns**
    - ['fills', 'no_fills', 'nombre_fills']
    - correct datatypes
    - validate coherence and null values in 'fills', 'no_fills'
    - combine the information so just 1 column remains de denote children
    - remove now redundant columns ('fills' and 'no_fills')

Initial evaluation of current data

In [93]:
df.describe(include = 'object')


,nom,cognoms,DNI,pais_d'origen,ciutat,genere,fills,no_fills,grup_professional,càrrec,te_cotxe
count,994,994,989,994,994,994,415,578,994,994,651
unique,102,467,989,5,44,2,1,2,4,9,2
top,Adrià,Ferrer,62042563B,Espanya,Palma,D,True,True,Grup A,Data Analyst,True
freq,34,21,1,709,58,504,415,577,462,157,326


In [94]:
# checking values (2) in no_fills:

grouped_fills_no = df.groupby('no_fills')
grouped_fills_no.agg({'employee_id': 'count'})

# checking values in fills:
grouped_fills_no = df.groupby('no_fills')
grouped_fills_no.agg({'employee_id': 'count'})


,employee_id
no_fills,
False,1
True,577


In [95]:
# checking data types for the 3 columns...
df[['fills', 'no_fills', 'nombre_fills']].info()


<class 'pandas.core.frame.DataFrame'>
Index: 994 entries, 0 to 1001
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   fills         415 non-null    object 
 1   no_fills      578 non-null    object 
 2   nombre_fills  698 non-null    float64
dtypes: float64(1), object(2)
memory usage: 63.4+ KB


Datatype needs changing for the 3 columns:  
- 'fills' ->> boolean
- 'no_fills' ->> boolean
- 'nombre_fills' ->> integer

#### fixing datatypes

In [96]:
# changing to bool and Int64 (rounding up decimals), applying to main df:

# first fill nulls with False in columns to convert to bool:
df['fills'] = df['fills'].fillna(False)
df['no_fills'] = df['no_fills'].fillna(False)

# now change data types all 3 columns ('nombre_fills' to IN64, then 'fills' and 'no_fills' to Boolean)
# to bool:
df['fills']= df['fills'].astype('bool')
df['no_fills'] = df['no_fills'].astype('bool')
# to INT64, rounding up decimal points, and filling missing values with 0:
df['nombre_fills'] = np.ceil(df['nombre_fills']).fillna(0).astype('Int64')





*->> got 'futurewarning' message, but changes applied ok. Now set pd.option at top so messages nolonger shown!*



```
pd.set_option("future.no_silent_downcasting", True)
```



In [97]:
# checking changes done are ok:

df[['fills', 'no_fills', 'nombre_fills']].describe(include = 'all')

,fills,no_fills,nombre_fills
count,994,994,994.00
unique,2,2,<NA>
top,False,True,<NA>
freq,579,577,<NA>
mean,NaN,NaN,0.84
...,...,...,...
min,NaN,NaN,0.00
25%,NaN,NaN,0.00
50%,NaN,NaN,0.00
75%,NaN,NaN,1.00


In [98]:
# checking for missing values in nombre_fills:  = OK, now 0 (was 296)
print(df['nombre_fills'].isna().sum())

0


#### Checking coherence of data over the 3 columns:

In [99]:

# checking inconsistencies first: same value in 'fills' and 'no_fills'...
incoherent_kids_allTrue = df.loc[(df['fills'] == True) & (df['no_fills'] == True)]
print(f"Number of employees with 'fills' and 'no_fills' both as True: {incoherent_kids_allTrue.employee_id.count()}")

incoherent_kids_allFalse = df.loc[(df['fills'] == False) & (df['no_fills'] == False)]
print(f"Number of employees with 'fills' and 'no_fills' both as False: {incoherent_kids_allFalse.employee_id.count()}")

incoherent_kids_True_0 = df.loc[(df['fills'] == True) & (df['nombre_fills'] == 0)]
print(f"Number of employees with 'fills' but 'nombre_fills' as 0: {incoherent_kids_True_0.employee_id.count()}")
incoherent_kids_False_Not0 = df.loc[(df['fills'] == False) & (df['nombre_fills'] != 0)]
print(f"Number of employees with 'no_fills' but 'nombre_fills' NOT as 0: {incoherent_kids_False_Not0.employee_id.count()}")

Number of employees with 'fills' and 'no_fills' both as True: 47
Number of employees with 'fills' and 'no_fills' both as False: 49
Number of employees with 'fills' but 'nombre_fills' as 0: 36
Number of employees with 'no_fills' but 'nombre_fills' NOT as 0: 14


### compiling data in single column

**Removing duplicity of data**  
3 columns in the dataset all refer to the parent status of the employees in the company.  

Having multiple columns representing the same condition opens the door to inconsistent and incoherent data due to the possibility of logically incompatible values in any single record. This can be seen by the summary above of incoherent data in our dataset.

Consequently, we will need to first clean-up the current data and then COMPILE FROM THE 3 ORIGINAL COLUMNS TO JUST 1, assuring no data is lost in the transformation.  

The column that provides the **most value in it's information is 'nombre_fills'**.

---


Reasoning:  
- The Y/N value I would loose by removing the 2 boolean columns ('fills' and 'no_fills') can be derived from the column with the count.
- If I removed the 'nombre_fills' column to just keep one of the boolean columns I would not be able to recuperate the count ever.
- There are less discrepancies between nombre_fills and each of the 2 boolean columns, so 'nombre_fills' suggests itself as a better source of truth
- Will put '1' in 'nombre_fills' for those that currently have 0 but have 'fills' as True (36 employees), so that they don't lose their parent status

Actions:  
1) 'fills' as True, but kids 0 in 'nombre_fills' (36 employees)  
  ->> PUT 1 in 'nombre_fills', as may have known a parent but not how many kids when data entered)
2) REMOVE 'fills' column as now redundant... (and have transfered/maintained employees' parent status in 'nombre-fills' column with above step)
3) REMOVE 'no_fills' column as now redundant... (and may have not been updated since employee hired anyways).

Using np.select, translate a True in 'Fills' to atleast 1 in 'nombre_fills'

In [100]:
# using np.select to translate a True in 'Fills' to atleast 1 in 'nombre_fills':

import numpy as np     # import also added at head of notebook

# Defining conditions and fixes, in order of priority
conditions = [# condition 1 (for which keep existing positive count)
              df['nombre_fills'] > 0,
              # condition 2 (has True but count is 0, so will put count as 1 in the fixes)
              (df['fills'] == True) & (df['nombre_fills'] == 0) ]

fixes = [# value condition 1 (value to keep)
        df['nombre_fills'],
        # value condition 2 (value to fix)
        1]

# Apply conditions (default to 0 for all other rows that don't meet the conditions)
df['nombre_fills'] = np.select(conditions, fixes, default=0).astype(int)

In [101]:
# checking results in main df:

incoherent_still_True_0 = df.loc[(df['fills'] == True) & (df['nombre_fills'] == 0)]
print(f"AFTER df FIX: Number of employees with 'fills' but 'nombre_fills' as 0: {incoherent_still_True_0.employee_id.count()}")

AFTER df FIX: Number of employees with 'fills' but 'nombre_fills' as 0: 0


Drop previous 2 boolean columns as now redundant

In [102]:
df = df.drop(columns=['fills', 'no_fills'])

In [103]:
# final check dropped, and also the count, values etc correct in remaining nombre-fills column.
df.describe(include = 'all')

,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,dia_de_naixement,mes_de_naixement,any_de_naixement,genere,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat,salari_mensual_def
count,994.00,994,994,989,994,994,994.00,994.00,994.00,994,994,994,994.00,651,313.00,259.00,928.00,994.00
unique,NaN,102,467,989,5,44,NaN,NaN,NaN,2,4,9,NaN,2,NaN,NaN,NaN,NaN
top,NaN,Adrià,Ferrer,62042563B,Espanya,Palma,NaN,NaN,NaN,D,Grup A,Data Analyst,NaN,True,NaN,NaN,NaN,NaN
freq,NaN,34,21,1,709,58,NaN,NaN,NaN,504,462,157,NaN,326,NaN,NaN,NaN,NaN
mean,500.60,NaN,NaN,NaN,NaN,NaN,15.14,8.49,2001.48,NaN,NaN,NaN,0.88,NaN,55952.87,11.05,15.36,1562.74
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
min,1.00,NaN,NaN,NaN,NaN,NaN,-5.00,0.00,1940.00,NaN,NaN,NaN,0.00,NaN,-50.00,0.50,-10.00,673.00
25%,251.25,NaN,NaN,NaN,NaN,NaN,7.00,3.25,1969.00,NaN,NaN,NaN,0.00,NaN,22384.00,4.65,12.80,1082.25
50%,499.50,NaN,NaN,NaN,NaN,NaN,15.00,6.00,1982.00,NaN,NaN,NaN,0.00,NaN,56135.00,9.80,15.50,1408.00
75%,749.75,NaN,NaN,NaN,NaN,NaN,24.00,10.00,1995.00,NaN,NaN,NaN,1.00,NaN,80000.00,13.65,18.30,1895.00


## DOB FIX

- **Date of Birth related columns**
    - ['dia_de_naixement', 'mes_de_naixement', 'any_de_naixement']
    - identify and evaluate impossible calendar values
    - evaluate implications of missing values
    - combine and convert data into single new column created, 'DOB' (date format)
    - remove now redundant columns (3 original columns)

#### Evaluating data and decision on how to transform

In [104]:
# saving all employees with bad DOB data (77 employees, aprox 7% of records)

bad_DOB = df.loc[(df['mes_de_naixement'] > 12.0) |
                (df['mes_de_naixement'] <= 0) |
                 (df['dia_de_naixement'] > 31.0) |
                 (df['dia_de_naixement'] <= 0) |
                    (df['any_de_naixement'] < 1956) | # would be over 70
                     (df['any_de_naixement'] > 2010), # would be under 16, legal working age with parental approval (also incl. future-dates)
                     ['employee_id', 'DNI', 'dia_de_naixement', 'mes_de_naixement', 'any_de_naixement']]
display(bad_DOB.sort_values('dia_de_naixement'))

,employee_id,DNI,dia_de_naixement,mes_de_naixement,any_de_naixement
47,47,38468360C,-5.00,0.00,3000.00
73,73,40097200A,-5.00,13.00,1940.00
175,175,81659221D,-5.00,99.00,3000.00
398,398,23482938N,-5.00,99.00,3000.00
268,268,99600963X,-5.00,13.00,2019.00
...,...,...,...,...,...
911,911,99290377X,32.00,13.00,3000.00
914,914,41786590C,32.00,99.00,2026.00
958,63,41407343V,32.00,99.00,2019.00
981,981,34747698K,32.00,0.00,2026.00



I need to convert the 3 columns for DOB to just one in datatime format, but many rows have impossible (day 32 of month, or year 3000, etc) or incomplete data in one or more of the 3 columns.  

As using coerce to correct these impossible calendar and missing values during the conversion would fill the DOB column with a NaN value, I had considered first adding a calculated column for approximate age, before converting and cleaning the final DOB column. This would mean that possible correct data in year column (age) would not be lost if just day or month data is a false value.  

HOWEVER, on further inspection, I see that **all rows with incorrect data in either day or month consistently have incorrect data in year**. This means I would not preserve any valuable data doing the aproximate age column first, so I can just compile the data from the 3 columns saving it in a **new datetime formatted 'DOB' column**.  
This also means I CAN use coerce (to put NaN in impossible values) as will not lose any valid data on age.

In [105]:
bad_DOB.describe()

,employee_id,dia_de_naixement,mes_de_naixement,any_de_naixement
count,77.00,77.00,77.00,77.00
mean,482.83,11.30,32.84,2245.26
std,290.59,16.76,42.48,436.04
min,9.00,-5.00,0.00,1940.00
25%,210.00,0.00,0.00,2019.00
50%,500.00,0.00,13.00,2019.00
75%,751.00,32.00,99.00,2026.00
max,996.00,32.00,99.00,3000.00


In [106]:
bad_DOB.groupby('any_de_naixement')['employee_id'].count()

,employee_id
any_de_naixement,
1940.00,17
2019.00,23
2026.00,18
3000.00,19


### Compile in new date-formatted 'DOB' column



Converting the 3 columns to datetime format and creating the new 'DOB' column.  
Using coerce for data errors, and mapping coulmn names from catalan so that the function recognizes them...

In [107]:
df['DOB'] = pd.to_datetime({
    'year': df['any_de_naixement'],
    'month': df['mes_de_naixement'],
    'day': df['dia_de_naixement']}, errors='coerce')

Now converted to date format and in a single variable, I can isolate and remove impossible ages/birthyears (replace with pd.NaT, 'Not a Time')

In [108]:
# calculate age to select impossible DOBs
current_year = pd.Timestamp.now().year
age = current_year - df['DOB'].dt.year

# select impossible ages (<16, >70, future date)
impossible_ages = (age < 16) | (age >= 70) | (df['DOB'] > pd.Timestamp.now())

# change impossible DOBs to NaT (Not a Time, which is NaN for date formatted columns)
df.loc[impossible_ages, 'DOB'] = pd.NaT

In [109]:
# checking df after fix to DOB applied: OK
display("DOB column describe:", df['DOB'].describe())
print("\n'Full 'df' info:' \n")
display(df.info())

display("DOB has NaT: ", df['DOB'].isnull().sum())

'DOB column describe:'

,DOB
count,917
mean,1981-07-01 06:31:00.850599808
min,1958-01-09 00:00:00
25%,1969-05-13 00:00:00
50%,1981-09-07 00:00:00
75%,1993-10-11 00:00:00
max,2004-12-27 00:00:00



'Full 'df' info:' 

<class 'pandas.core.frame.DataFrame'>
Index: 994 entries, 0 to 1001
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   employee_id                 994 non-null    int64         
 1   nom                         994 non-null    object        
 2   cognoms                     994 non-null    object        
 3   DNI                         989 non-null    object        
 4   pais_d'origen               994 non-null    object        
 5   ciutat                      994 non-null    object        
 6   dia_de_naixement            994 non-null    float64       
 7   mes_de_naixement            994 non-null    float64       
 8   any_de_naixement            994 non-null    float64       
 9   genere                      994 non-null    object        
 10  grup_professional           994 non-null    object        
 11  càrrec                      994 non-null 

None

'DOB has NaT: '

np.int64(77)

Confirmed new column correctly created populated, can now remove original day/month/year columns from the dataset to prevent ambiguity.

In [110]:
df = df.drop(columns=['dia_de_naixement', 'mes_de_naixement', 'any_de_naixement'])

In [111]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 994 entries, 0 to 1001
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   employee_id                 994 non-null    int64         
 1   nom                         994 non-null    object        
 2   cognoms                     994 non-null    object        
 3   DNI                         989 non-null    object        
 4   pais_d'origen               994 non-null    object        
 5   ciutat                      994 non-null    object        
 6   genere                      994 non-null    object        
 7   grup_professional           994 non-null    object        
 8   càrrec                      994 non-null    object        
 9   nombre_fills                994 non-null    int64         
 10  te_cotxe                    651 non-null    object        
 11  km_anuals                   313 non-null    float64       
 12

## TEMPERATURE FIX

- **'temperatura_mitja_ciutat'**
    - evaluate coherence of values
    - standardize values per city


### evaluating data and decision on transformation

In [112]:
# seeing what values we have:

display("describe data: ", df['temperatura_mitjana_ciutat'].describe())
print()
display("rows missing temperature data: ", df['temperatura_mitjana_ciutat'].isna().sum())
# display(df['ciutat'].loc[df.nsmallest(1, 'temperatura_mitjana_ciutat')])
print()
display("have very low value (incorrect): ", df[['ciutat', 'temperatura_mitjana_ciutat']].sort_values('temperatura_mitjana_ciutat').head(10))
display(df[['ciutat', 'temperatura_mitjana_ciutat']].loc[df['temperatura_mitjana_ciutat'] < 10])


'describe data: '

,temperatura_mitjana_ciutat
count,928.00
mean,15.36
std,7.26
min,-10.00
25%,12.80
50%,15.50
75%,18.30
max,42.00


'rows missing temperature data: '

np.int64(66)

'have very low value (incorrect): '

,ciutat,temperatura_mitjana_ciutat
999,Palma,-10.00
317,Palma,-10.00
436,Napoli,-10.00
885,Las Palmas de Gran Canaria,-10.00
513,València,-10.00
887,Alacant,-10.00
787,Barcelona,-10.00
240,Frankfurt,-10.00
197,Düsseldorf,-10.00
148,Zaragoza,-10.00


,ciutat,temperatura_mitjana_ciutat
8,Stavanger,7.60
17,Bergen,8.80
28,Sevilla,-5.00
63,Trondheim,7.00
81,Valladolid,9.70
...,...,...
963,Madrid,8.60
974,Roma,-10.00
988,Girona,9.50
999,Palma,-10.00


In [113]:
city_by_temp = df.groupby('ciutat')['temperatura_mitjana_ciutat'].agg(['count','nunique', 'mean', 'median', 'min', 'max'])

display(city_by_temp)

,count,nunique,mean,median,min,max
ciutat,,,,,,
A Coruña,44,32,12.94,13.85,-5.00,19.30
Alacant,40,33,17.37,17.95,-10.00,42.00
Barcelona,51,40,16.78,16.60,-10.00,42.00
Bergen,10,9,8.82,7.85,-5.00,42.00
Berlin,6,5,9.53,9.85,7.60,11.10
...,...,...,...,...,...,...
Trondheim,6,5,4.38,3.95,2.40,7.00
Valladolid,43,36,13.41,12.20,-5.00,42.00
València,32,28,16.65,17.30,-10.00,21.30


**Observations on average temperature per city data present**  
As is evident from the above summary tables, not counting missing (NaN) values, each of the 44 cities in the dataframe have many different values reported for their average temperature.  

Therefore I have decided to follow the rule **"same city same value"**, throughout:  

- calculate the median value for each city (to reduce influence of extreme values)  
- replace the value for ALL rows/cities, independant to whether that row already had a temperature recorded

This will consolidate the data into **logical values not only for each row but also as a whole**.

### fixing temperature for each city

Although the action needed is similar to the fix I applied to the salary column (impute missing values with the median value), here, as I am modifying **whole series**, I can use `.transform()`, which doesn't need an intermediate step of creating a dict of the median values (calculates them directly in the one line of code).

In [114]:
# applying average temerature fix to main df:
df['temperatura_mitjana_ciutat'] = (df.groupby('ciutat')['temperatura_mitjana_ciutat'].transform('median').round(1))


check each city has 1 for `nunique` on the temperature column

In [115]:
fixed_temps = df.groupby('ciutat')['temperatura_mitjana_ciutat'].agg(['count','nunique', 'mean', 'median', 'min', 'max'])

display(fixed_temps.sort_values('nunique', ascending = False))

,count,nunique,mean,median,min,max
ciutat,,,,,,
A Coruña,46,1,13.80,13.80,13.80,13.80
Alacant,44,1,18.00,18.00,18.00,18.00
Barcelona,54,1,16.60,16.60,16.60,16.60
Bergen,10,1,7.80,7.80,7.80,7.80
Berlin,6,1,9.80,9.80,9.80,9.80
...,...,...,...,...,...,...
Trondheim,6,1,4.00,4.00,4.00,4.00
Valladolid,44,1,12.20,12.20,12.20,12.20
València,39,1,17.30,17.30,17.30,17.30


## CAR / KM FIXES

- **Vehicle related columns**
    - ['te_cotxe', 'km_anuals', 'consum_mitja_l_100km']
    - check coherence between the 3 columns and validate data
    - correct datatypes
    - identify extreme values (outliers) and negative values (to NaN)
    - evaluate imputation options
    - imputate missing values

### exploring data and decisions on coherency and missing values

Checking how many rows have data, and the data types:

In [116]:
df[['te_cotxe', 'km_anuals', 'consum_mitja_l_100km']].info()

<class 'pandas.core.frame.DataFrame'>
Index: 994 entries, 0 to 1001
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   te_cotxe              651 non-null    object 
 1   km_anuals             313 non-null    float64
 2   consum_mitja_l_100km  259 non-null    float64
dtypes: float64(2), object(1)
memory usage: 63.4+ KB


In [117]:
df[['te_cotxe', 'km_anuals', 'consum_mitja_l_100km']].describe(include = 'all')

,te_cotxe,km_anuals,consum_mitja_l_100km
count,651,313.00,259.00
unique,2,NaN,NaN
top,True,NaN,NaN
freq,326,NaN,NaN
mean,NaN,55952.87,11.05
...,...,...,...
min,NaN,-50.00,0.50
25%,NaN,22384.00,4.65
50%,NaN,56135.00,9.80
75%,NaN,80000.00,13.65


Checking how many have a car, but are missing km or fuel consumption details: = 69

In [118]:
incoherent_car_only = df.loc[(df['te_cotxe'] == True) & (df['km_anuals'].isna() | df['consum_mitja_l_100km'].isna())]

print(f"{incoherent_car_only['employee_id'].count()} employees recorded as having a car, but either missing km or fuel info:")
display(incoherent_car_only[['te_cotxe', 'km_anuals', 'consum_mitja_l_100km']])


69 employees recorded as having a car, but either missing km or fuel info:


,te_cotxe,km_anuals,consum_mitja_l_100km
4,True,NaN,11.80
15,True,65950.00,NaN
26,True,NaN,NaN
47,True,78064.00,NaN
58,True,65950.00,NaN
...,...,...,...
969,True,7861.00,NaN
977,True,97162.00,NaN
985,True,56745.00,NaN
989,True,27921.00,NaN


Checking how many have km or fuel consumption details, but not reported as having a car: = 0

In [119]:
# runnning this code retuens 0 employees (without car but fuel or km details):
incoherent_km_fuel_only = df.loc[(df['te_cotxe'] != True) & (df['km_anuals'] + df['consum_mitja_l_100km'] > 0)]
display(incoherent_km_fuel_only)

# double-checking really none with fuel and km details that don't have a car: = OK
df.loc[df['te_cotxe'] != True][['te_cotxe', 'km_anuals', 'consum_mitja_l_100km']].sort_values(['km_anuals', 'consum_mitja_l_100km'], ascending = False)

,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,genere,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat,salari_mensual_def,DOB


,te_cotxe,km_anuals,consum_mitja_l_100km
270,False,0.00,NaN
698,False,0.00,NaN
743,False,0.00,NaN
922,False,0.00,NaN
2,NaN,NaN,NaN
...,...,...,...
995,False,NaN,NaN
996,NaN,NaN,NaN
998,NaN,NaN,NaN
999,False,NaN,NaN


**Coherence / missing data:**  
There is missing data (employees that have a car but no data in 'km_anuals' and/or in 'consum_mitja_l_km'), but there are not employees with fuels and km data that are missing being flagged as having a car.   

Given the instructions *"... consum mitjà però no quilometratge anual, caldrà decidir com resoldre aquesta incoherència de manera justificada"*, I need to decide how to impute these missing values.

Also, in order to avoid skewed data for future analysis of the vehicle statistics, employees without a car should have NaN for the 2 numerical columns (and not 0 as they have currently).   


**Data types**  
The 'te_cotxe' variable is an object. Convert to boolean type, making sure no data lost.   

Both 'km_anuals' and 'consum_mitja_l_100km' are float.  
This is acceptable, eventhough 'km_anuals' could be integer (as decimal details is not necessary or not influential when concerning such large numbers), I have decided that it is preferential that both variables have the same numeric format in preparation for possible future analysis calaculations.  





### fixing datatypes and 0 values

#### 'te_cotxe'  


(BEFORE STATE) - Currently is an object, and has NaN values

In [120]:
print(f"BEFORE FIX data type of car column: {df['te_cotxe'].dtypes}")
print(f"BEFORE FIX employees WITH car: {df['te_cotxe'].loc[df['te_cotxe'] == True].count()}")
print(f"BEFORE FIX values in car column: {df['te_cotxe'].nunique()}\n")

BEFORE FIX data type of car column: object
BEFORE FIX employees WITH car: 326
BEFORE FIX values in car column: 2



(FIX) - convert NaN to False and datatype to boolean

In [121]:
df['te_cotxe'] = df['te_cotxe'].fillna(False).astype('bool')

(AFTER STATE) - re_running above checks to see if change ok and no data lost

In [122]:
print(f"AFTER FIX data type of car column: {df['te_cotxe'].dtypes}")
print(f"AFTER FIX employees WITH car: {df['te_cotxe'].loc[df['te_cotxe'] == True].count()}")
print(f"AFTER FIX values in car column: {df['te_cotxe'].nunique()}")

AFTER FIX data type of car column: bool
AFTER FIX employees WITH car: 326
AFTER FIX values in car column: 2


#### 'km_anuals'  


(BEFORE STATE) - Convert 0 to NaN values

In [123]:
# checking NaN values and 0 values:
print(f"\nkm value as NaN: {df['km_anuals'].isna().sum()}")
print(f"km value as 0: {df.loc[df['km_anuals'] == 0]['employee_id'].count()}")

# checking if any of those with km value 0 has te_cotxe True? = no = OK
print('any of those with km value 0 has te_cotxe True? = no...')
display(df[['te_cotxe', 'km_anuals', 'consum_mitja_l_100km']].loc[df['km_anuals'] == 0])


km value as NaN: 681
km value as 0: 4
any of those with km value 0 has te_cotxe True? = no...


,te_cotxe,km_anuals,consum_mitja_l_100km
270,False,0.00,NaN
698,False,0.00,NaN
743,False,0.00,NaN
922,False,0.00,NaN


(FIX) - converting 0 to NaN

In [124]:
df['km_anuals'] = df['km_anuals'].replace(0, np.nan)

(AFTER STATE) - running checks to see fix ok

In [125]:
# checking fix worked: OK
print('\nkm_anuals info')
df['km_anuals'].info()

# checking NaN values and 0 values changed: OK
print(f"\n AFTER FIX - km value as NaN: {df['km_anuals'].isna().sum()}")
print(f"AFTER FIX - km value as 0: {df.loc[df['km_anuals'] == 0]['employee_id'].count()}")


km_anuals info
<class 'pandas.core.series.Series'>
Index: 994 entries, 0 to 1001
Series name: km_anuals
Non-Null Count  Dtype  
--------------  -----  
309 non-null    float64
dtypes: float64(1)
memory usage: 47.8 KB

 AFTER FIX - km value as NaN: 685
AFTER FIX - km value as 0: 0


#### 'consum_mitja_l_100km'  


(OK) - - Has no 0 values that need converting

In [126]:
# checking fuel data
print('\nconsum_mitja_l_100km info')
df['consum_mitja_l_100km'].info()

# checking NaN values and 0 values
print(f"\n CURRENT STATE - fuel value as NaN: {df['consum_mitja_l_100km'].isna().sum()}")
print(f"CURRENT STATE - fuel value as 0: {df.loc[df['consum_mitja_l_100km'] == 0]['employee_id'].count()}")


consum_mitja_l_100km info
<class 'pandas.core.series.Series'>
Index: 994 entries, 0 to 1001
Series name: consum_mitja_l_100km
Non-Null Count  Dtype  
--------------  -----  
259 non-null    float64
dtypes: float64(1)
memory usage: 47.8 KB

 CURRENT STATE - fuel value as NaN: 735
CURRENT STATE - fuel value as 0: 0


### Identify Outliers and excessive values


Before any imputation strategies can be studied, data inconsisties need to be identified and corrected:  
- Replace impossible numbers (excessively high, or negative values) with NaN.  
- Fix any inconsistencies with decimal point position on both numerical columns.  



#### FUNCTION to detect outliers (to then replace with NaN)

As **identifying outliers** in numeric variables is a common task in data analytics, I have created a **resusable function** that can be applied to any numeric variable and it will return a filtered copy of the dataset with the outliers that have been detected.

---
notes on IDENTIFYING OUTLIERS:

- **Range** = max - min
- **IQR** =  Q3 - Q1
- **OUTLIERS** =  
    >> value > (Q3 + (1.5* IQR))  
    >> value < (Q1 + (1.5 * IQR))

---

**FUNCTION: detect_outliers()**  
- INPUT: dataset, col_name (str, name of column to be evaluated)
- calculates the interquartile ranges (Q1 and Q3) for the selected column
- defines the upper and lower limits to flag any outliers
- selects the rows with values above or below the limits set
- RETURN: filtered DF with rows detected as having outlier values in the column defined

In [127]:
def detect_outliers(dataset, col_name):

  # calculate Q1, Q3 and IQR:
  Q1 = dataset[col_name].quantile(0.25)
  Q3 = dataset[col_name].quantile(0.75)
  IQR = Q3 - Q1

  # define limits to detect outliers:
  lower_limit = Q1 - (1.5 * IQR)
  upper_limit = Q3 + (1.5 * IQR)

  # select the outliers
  outliers_detected = dataset[(dataset[col_name] < lower_limit) | (dataset[col_name] > upper_limit)]

  # return
  return outliers_detected


#### converting detected inconsistencies to NaN

Select outliers (KM and Fuel) in df using outliers function:

In [128]:
#calling function for each of the columns:
km_outliers = detect_outliers(df, 'km_anuals')
fuel_outliers = detect_outliers(df, 'consum_mitja_l_100km')

check results

In [129]:
# showing results:
print("\nOutliers detected in df > 'km_anuals'")
display(km_outliers)

print("\nOutliers detected in df > 'consum_mitja_l_100km'")
display(fuel_outliers)


Outliers detected in df > 'km_anuals'


,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,genere,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat,salari_mensual_def,DOB
213,213,Oriol,Díaz Romero,10968781V,Espanya,Tarragona,H,Grup A,Administratiu,0,True,1000000.00,25.00,16.80,935.00,1996-04-26



Outliers detected in df > 'consum_mitja_l_100km'


,employee_id,nom,cognoms,DNI,pais_d'origen,ciutat,genere,grup_professional,càrrec,nombre_fills,te_cotxe,km_anuals,consum_mitja_l_100km,temperatura_mitjana_ciutat,salari_mensual_def,DOB


Select negative values in both columns (KM and Fuel) to include in the Convert to NaN action.

In [130]:
# >>> negative values in KM:
km_negative = df.loc[df['km_anuals'] < 0]

# >>> negative values in Fuel (none, but will include so complete solution if data changes in future):
fuel_negative = df.loc[df['consum_mitja_l_100km'] < 0]


In [131]:
print(f" negative values detected in km_anuals: {km_negative['employee_id'].count()}")
print(f" negative values detected in consum_mitja_l_100km: {fuel_negative['employee_id'].count()}")

 negative values detected in km_anuals: 6
 negative values detected in consum_mitja_l_100km: 0


Use the 'join' function `.union()` to combine the above selections into an index list, which I can then use to replace outliers and negative values in the columns with NaN (KM and Fuel):

In [132]:
# >>>> replacing with NaN in KM:

# use JOIN on above filtered df copies to get an index list of all the rows selected (returns sets of row indexes, not full filtered df):
km_to_nan = km_outliers.index.union(km_negative.index)

# use .loc with the index list as the condition to replace km values in df:
df.loc[km_to_nan, 'km_anuals'] = np.nan

# >>>> replacing with NaN in Fuel :

# use JOIN on above filtered df copies to get an index list of all the rows selected:
fuel_to_nan = fuel_outliers.index.union(fuel_negative.index)

# use .loc with the index list as the condition to replace fuel values in df:
df.loc[fuel_to_nan, 'consum_mitja_l_100km'] = np.nan


checking the fix has corrected all outlier values detetected (1) and negative values (4)

In [133]:
# >>> OUTLIERS FIXED?
# re-run outliers function on km, which before had 1 value identified. should now identify none = OK
checking_km_fix = detect_outliers(df, 'km_anuals')
print("outliers detected after fix (km): ", checking_km_fix['employee_id'].count())

# >>> NEGATIVE VALUES FIXED?
checking_km_negative_fix = df.loc[df['km_anuals'] < 0]
print("negative values after fix (km): ", checking_km_negative_fix['employee_id'].count())

# >>>> HOW MANY NaN VALUES NOW?
# checking NaN values and 0 values:
print(f"\nkm value as NaN: {df['km_anuals'].isna().sum()} (before fix, count was 681)")
print(f"km value as 0: {df.loc[df['km_anuals'] == 0]['employee_id'].count()} (before fix, count was 4)")

outliers detected after fix (km):  0
negative values after fix (km):  0

km value as NaN: 692 (before fix, count was 681)
km value as 0: 0 (before fix, count was 4)


### Evaluation of imputation of missing values

In order to decide on the best way to imputate missing values I need to:  
- study possible km/fuel consumption indicators, what factors are relevant?  
- for example country/carrec?, or relationship between consumption and average km?

In [134]:
have_car = df.loc[df['te_cotxe'] == True]

have_car[['te_cotxe', 'km_anuals', 'consum_mitja_l_100km']].describe(include = 'all')

,te_cotxe,km_anuals,consum_mitja_l_100km
count,326,302.00,259.00
unique,1,NaN,NaN
top,True,NaN,NaN
freq,326,NaN,NaN
mean,NaN,54680.62,11.05
...,...,...,...
min,NaN,50.00,0.50
25%,NaN,26883.50,4.65
50%,NaN,60079.50,9.80
75%,NaN,80000.00,13.65


checking partial data that also needs completing

In [135]:
# need to check if km_anuals that are missing fuel data and viceversa... (= 49 incomplete rows with one or other only)

km_no_fuel = df.loc[(df['km_anuals'] > 0) & (df['consum_mitja_l_100km'].isnull())]
print(f"km data but no fuel data: {km_no_fuel['employee_id'].count()}")

fuel_no_km = df.loc[(df['consum_mitja_l_100km'] > 0) & (df['km_anuals'].isnull())]
print(f"fuel data but no km data: {fuel_no_km['employee_id'].count()}")

#just checking all these are recorded as having a car: = OK
print(km_no_fuel['employee_id'][km_no_fuel['te_cotxe'] == False].count())
print(fuel_no_km['employee_id'][fuel_no_km['te_cotxe'] == False].count())

km data but no fuel data: 46
fuel data but no km data: 3
0
0


#### TESTS: Numeric values correlate?

Use **Pearson** (linear relationships) and **Spearman** (monotonic relationship) correlation tests to see if any numerical columns are relatable to our km and fuel data.

In [136]:
# numeric columns to check:
numeric_cols = ['km_anuals', 'consum_mitja_l_100km', 'salari_mensual_def', 'nombre_fills', 'temperatura_mitjana_ciutat']

# Pearson correlation (linear relationships)
corr_matrix = df[numeric_cols].corr()
print(" ->> Pearson Correlation <<-")
display(corr_matrix[['km_anuals', 'consum_mitja_l_100km']])

# Spearman correlation (monotonic relationships, robust against non-linear patterns)
spearman_corr = df[numeric_cols].corr(method='spearman')
print("\n->> Spearman Correlation <<-")
display(spearman_corr[['km_anuals', 'consum_mitja_l_100km']])


 ->> Pearson Correlation <<-


,km_anuals,consum_mitja_l_100km
km_anuals,1.00,0.09
consum_mitja_l_100km,0.09,1.00
salari_mensual_def,-0.08,0.00
nombre_fills,-0.03,0.06
temperatura_mitjana_ciutat,-0.07,-0.07



->> Spearman Correlation <<-


,km_anuals,consum_mitja_l_100km
km_anuals,1.00,0.08
consum_mitja_l_100km,0.08,1.00
salari_mensual_def,-0.05,0.00
nombre_fills,0.05,0.07
temperatura_mitjana_ciutat,-0.06,-0.03


---

INTERPRETATION:  

- **> 0.3** indicates a **moderate** relationship  
- **> 0.5** indicates a **strong** relationship  
- **< 0.1** **low** correlation

---

The above results suggest that there is **NO CORRELATION** between any of the **numeric columns** in our dataset ('salari_mensual', 'nombre_fills', 'temperatura_mitjana_ciutat') and the values we have in the 2 vehicle statistics series that we need to complete ('km_anuals', 'consum_mitja_l_100km').

#### TESTS: Categorical values correlate?

TEst **variance** in minimum and maximum **median values**, in order to identify if any categorical variables correlate to the km and the fuel data


---

INTERPRETATION:  
- **highest variance = most related**  


Meaning that, **higher variance** between min and max group medians denotes **higher correlation probability**, translating into being the better the predictor.  

---

(NB: add calculation of differences to the loop that runs the test)

In [137]:
df[['km_anuals', 'consum_mitja_l_100km']].describe()

,km_anuals,consum_mitja_l_100km
count,302.00,259.00
mean,54680.62,11.05
std,30844.03,8.80
min,50.00,0.50
25%,26883.50,4.65
50%,60079.50,9.80
75%,80000.00,13.65
max,118147.00,25.00


In [138]:
# categorical columns to check:
categ_columns = ["pais_d'origen", "ciutat", "grup_professional", "càrrec"]

print("->> KM >>>:")
# Check how km_anuals varies across the different categorical columns:
for cat in categ_columns:
  role_stats_km = df.groupby(cat)['km_anuals'].agg(['count', 'median', 'mean', 'std', 'max', 'min'])
  # Calculate difference between max and min median
  max_med_km = role_stats_km['median'].max()
  min_med_km = role_stats_km['median'].min()
  diff_med_km = max_med_km - min_med_km
  # print summary and full role stats:
  print(f"\n--- Category compared: {cat} ---")
  print(f"Max Median: {max_med_km:,.1f} | Min Median: {min_med_km:,.1f} | Difference: {diff_med_km:,.1f}")

print("\n\n->> FUEL >>>:")
# Check how consum_mitja_l_100km varies across the different categorical columns:
for cat in categ_columns:
  role_stats_fuel = df.groupby(cat)['consum_mitja_l_100km'].agg(['count', 'median', 'mean', 'std', 'max', 'min'])
  # Calculate difference between max and min median
  max_med_fuel = role_stats_fuel['median'].max()
  min_med_fuel = role_stats_fuel['median'].min()
  diff_med_fuel = max_med_fuel - min_med_fuel
  # print summary and full role stats:
  print(f"\n--- Category compared: {cat} ---")
  print(f"Max Median: {max_med_fuel:,.1f} | Min Median: {min_med_fuel:,.1f} | Difference: {diff_med_fuel:,.1f}")


->> KM >>>:

--- Category compared: pais_d'origen ---
Max Median: 80,000.0 | Min Median: 56,135.0 | Difference: 23,865.0

--- Category compared: ciutat ---
Max Median: 82,167.0 | Min Median: 20,000.0 | Difference: 62,167.0

--- Category compared: grup_professional ---
Max Median: 60,856.0 | Min Median: 35,591.0 | Difference: 25,265.0

--- Category compared: càrrec ---
Max Median: 65,966.5 | Min Median: 40,000.0 | Difference: 25,966.5


->> FUEL >>>:

--- Category compared: pais_d'origen ---
Max Median: 10.3 | Min Median: 6.8 | Difference: 3.5

--- Category compared: ciutat ---
Max Median: 25.0 | Min Median: 0.5 | Difference: 24.5

--- Category compared: grup_professional ---
Max Median: 12.9 | Min Median: 9.1 | Difference: 3.8

--- Category compared: càrrec ---
Max Median: 12.2 | Min Median: 6.5 | Difference: 5.7


The results suggest that the single categorical variables with most correlation to our vehicle statistics would be **'ciutat' and 'càrrec'**, for both the KM and the Fuel data.

Check if ciutat and carrec combined gives more reliable basis for imputation.  



In [139]:
# Group by both ciutat and carrec >> KM:
print("\nKM >>>>")
combo_stats_km = df.groupby(['càrrec', 'ciutat'])['km_anuals'].agg(['count', 'median'])
# Calculate difference between max and min median
max_med_combkm = combo_stats_km['median'].max()
min_med_combkm = combo_stats_km['median'].min()
diff_med_combkm = max_med_combkm - min_med_combkm
# print summary and full role stats:
print(f"\n--- groupedby 'càrrec' & 'ciutat' ---")
print(f"Max Median: {max_med_combkm:,.1f} | Min Median: {min_med_combkm:,.1f} | Difference: {diff_med_combkm:,.1f}")

# Group by both ciutat and carrec >> FUEL:
print("\nFUEL >>>>")
combo_stats_fuel = df.groupby(['càrrec', 'ciutat'])['consum_mitja_l_100km'].agg(['count', 'median'])
# Calculate difference between max and min median
max_med_combfuel = combo_stats_fuel['median'].max()
min_med_combfuel = combo_stats_fuel['median'].min()
diff_med_combfuel = max_med_combfuel - min_med_combfuel
# print summary and full role stats:
print(f"\n--- groupedby 'càrrec' & 'ciutat' ---")
print(f"Max Median: {max_med_combfuel:,.1f} | Min Median: {min_med_combfuel:,.1f} | Difference: {diff_med_combfuel:,.1f}")



KM >>>>

--- groupedby 'càrrec' & 'ciutat' ---
Max Median: 118,147.0 | Min Median: 4,934.0 | Difference: 113,213.0

FUEL >>>>

--- groupedby 'càrrec' & 'ciutat' ---
Max Median: 25.0 | Min Median: 0.5 | Difference: 24.5



The results suggest that, yes, the combined group gives more variance for KM data (from 62k or 26k measuring the single variables, to 113K when combined), and the results for fuel data remains the same (still 24.5).

**Final imputation strategy:**  

**Values to use for imputation of missing values in Km and fuel consumption** (for those employees with car but with either of these columns empty):  

- None of the data in any of the numeric columns shows any correlation to the values in either the km or the fuel columns.  

- After running tests on the various categorical variables, I see that the variable most probable to influence both values needing imputation is **'ciutat'**, followed by **'càrrec'**.  

- Using a **combination of both of these variables** to calculate the median values suggest even higher relationship (higher variance in minimum and maximum median values  

I will **use the median values from the 2-level grouping aggregation, of 'ciutat' and 'càrrec', for my imputations**.  

### filling missing values

#### imputating KM missing data

(BEFORE STATE)

In [140]:
car_no_km_count = df.loc[(df['km_anuals'].isna()) & (df['te_cotxe'] == True)]['employee_id'].count()

print("BEFORE...")
print(f"has car & no km data: {car_no_km_count}")


BEFORE...
has car & no km data: 24


Filtering only those with car, to use for the reference median values for imputation

In [141]:
# filtered copy, only those with car:
car_yes = df.loc[df['te_cotxe'] == True]

Use the above filtered copy of those with car to calculate the median values and create the dictionary of values to map.  

Also create a **back-up dictionary** (medians of the just the single variable 'carrec', incase there is no data for any specific combination of 'carrec' and 'ciutat' that needs filling).  



In [142]:
# select the rows with missing km data:
km_missing = car_yes['km_anuals'].isna()

# calculate reference medians (combined categories) for aggregated data:
km_car_yes_medians = car_yes.groupby(["càrrec", "ciutat"])[['km_anuals']].agg('median')

# construct dictionary to use in map() when applying fix to missing km:
km_medians_dict = km_car_yes_medians.to_dict()

# construct safety-net dictionary of medians (only 'càrrec') for those combinations without values to map:
safetynet_càrrec_dict = car_yes.groupby('càrrec')['km_anuals'].median().to_dict()

Zip the 'càrrec' and 'ciutat' series' in order to make the combined **lookup key** for the nested dictionary.  

NB: I must save the result as a list, so that it is always available.

In [143]:
km_combined_key = list(zip(df['càrrec'], df['ciutat']))

# nb: can't use the filtered 'yes_car' copy as i get a valueError (different number of rows) in the map...


Use the zipped list of tuples (càrrec & ciutat combinations in the dataset) for applying the `map()` action within the `fillna` operation on the km column.  


I am chaining afterwards `fillna` again to catch any remaining NaN with medians that need to be taken from the safety_net dictionary (càrrec only grouping).

In [144]:
df['km_anuals'] = (df['km_anuals']
  .fillna(pd.Series(km_combined_key, index=df.index).map(km_medians_dict['km_anuals']))
  .fillna(df['càrrec'].map(safetynet_càrrec_dict)))

(AFTER STATE) - fix to km: NaN values replaced with median from combined grouping of carrec and ciutat



In [145]:
AFTER_car_no_km_count = df.loc[(df['km_anuals'].isna()) & (df['te_cotxe'] == True)]['employee_id'].count()

print("AFTER...")
print(f"has car & no km data: {AFTER_car_no_km_count}")

AFTER...
has car & no km data: 0


#### imputating FUEL missing data

(BEFORE STATE)

In [146]:
car_no_fuel_count = df.loc[(df['consum_mitja_l_100km'].isna()) & (df['te_cotxe'] == True)]['employee_id'].count()

print("BEFORE...")
print(f"has car & no fuel data: {car_no_fuel_count}")


BEFORE...
has car & no fuel data: 67


Use same filtered copy of only those with car ('car_yes') for the reference median values for imputation

In [147]:
# filtered copy, only those with car:
# car_yes = df.loc[df['te_cotxe'] == True]

Creation of dictionary, and back-up dictionary, of values to map.  


In [148]:
# select the rows with missing km data:
fuel_missing = car_yes['consum_mitja_l_100km'].isna()

# calculate reference medians (combined categories) for aggregated data:
fuel_car_yes_medians = car_yes.groupby(["càrrec", "ciutat"])[['consum_mitja_l_100km']].agg('median')

# construct dictionary to use in map() when applying fix to missing km:
fuel_medians_dict = fuel_car_yes_medians.to_dict()

# construct safety-net dictionary of medians (only 'càrrec') for those combinations without values to map:
fuel_safetynet_càrrec_dict = car_yes.groupby('càrrec')['consum_mitja_l_100km'].median().to_dict()

Use the same combined **lookup key** for the nested dictionary.  


In [149]:
# km_combined_key = list(zip(df['càrrec'], df['ciutat']))

Apply the `map()` action within the chained `fillna` operations on the fuel column

In [150]:
df['consum_mitja_l_100km'] = (df['consum_mitja_l_100km']
  .fillna(pd.Series(km_combined_key, index=df.index).map(fuel_medians_dict['consum_mitja_l_100km']))
  .fillna(df['càrrec'].map(fuel_safetynet_càrrec_dict)))

(AFTER STATE) - fix to fuel data: NaN values replaced with median from combined grouping of carrec and ciutat



In [151]:
AFTER_car_no_fuel_count = df.loc[(df['consum_mitja_l_100km'].isna()) & (df['te_cotxe'] == True)]['employee_id'].count()

print("AFTER...")
print(f"has car & no fuel data: {AFTER_car_no_fuel_count}")

AFTER...
has car & no fuel data: 0


# **final Dataframe** after transformation (**df_final**)

Recuperate the original column order and column titles, saving the cleaned dataframe as 'df_final'

In [152]:
df.columns

Index(['employee_id', 'nom', 'cognoms', 'DNI', 'pais_d'origen', 'ciutat',
       'genere', 'grup_professional', 'càrrec', 'nombre_fills', 'te_cotxe',
       'km_anuals', 'consum_mitja_l_100km', 'temperatura_mitjana_ciutat',
       'salari_mensual_def', 'DOB'],
      dtype='object')

Change the name of new salary column created (salari_mensual_def) to the title of the original column, 'salari_mensual'.

In [153]:
df = df.rename(columns={'salari_mensual_def':'salari_mensual'})

Save the final copy, reordering the columns to match the order of the original dataframe imported

In [154]:
df_final = df[['employee_id', 'nom', 'cognoms', 'DNI', "pais_d'origen", 'ciutat', 'DOB',
       'genere', 'salari_mensual', 'grup_professional', 'càrrec', 'nombre_fills', 'te_cotxe',
       'km_anuals', 'consum_mitja_l_100km', 'temperatura_mitjana_ciutat']]

Info on the transformed dataframe that is now saved, ready for future analysis:

In [155]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 994 entries, 0 to 1001
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   employee_id                 994 non-null    int64         
 1   nom                         994 non-null    object        
 2   cognoms                     994 non-null    object        
 3   DNI                         989 non-null    object        
 4   pais_d'origen               994 non-null    object        
 5   ciutat                      994 non-null    object        
 6   DOB                         917 non-null    datetime64[ns]
 7   genere                      994 non-null    object        
 8   salari_mensual              994 non-null    float64       
 9   grup_professional           994 non-null    object        
 10  càrrec                      994 non-null    object        
 11  nombre_fills                994 non-null    int64         
 12

# **NIVELL 1 - EXERCISE 4**  
---


#### **Primeres anàlisis descriptives**

***Objectiu**: obtenir una primera visió global del dataset un cop s'han aplicat les transformacions bàsiques, mitjançant estadístiques descriptives i comparacions simples entre grups rellevants.*  

***Aspectes a revisar i analitzar:***

» *Construir taules resum descriptives.*  
» *Fer comparacions bàsiques entre grups rellevants.*  

***Concretament, hauràs de:***  

*» Construir taules resum descriptives de salaris com una taula resum que permeti veure el sou mig, medià, mínim i màxim per Gènere. Ordena la taula en funció del sou mig.*  

» *Crear una taula resum amb el salari mig per gènere (files) i país d'origen (columnes). Afegeix-hi les mitjanes als marges de la taula. Aplica format condicional a la taula per veure en un color més intens els valors més elevats.*  

___


###**Table 1:** summarizing **key descriptive statistics of monthly salary** ('salari_mensual') **by gender**:

In [156]:
gender_wages = df_final.groupby('genere')['salari_mensual'].agg(['mean', 'median','min', 'max', 'count'])
display(gender_wages.sort_values('mean', ascending = False))

,mean,median,min,max,count
genere,,,,,
D,1577.45,1433.50,722.00,3672.00,504
H,1547.61,1380.00,673.00,3672.00,490


From the summary provided in the above table, measuring the **average, median, minimum and maximum monthly salaries** when employee records are grouped **by gender**, we can see that:  

- **Female employees make up just over half of the workforce** (504 women compared to 490 men)  

- **Women's average monthly wage is marginally higher** than their male co-workers' (roughly 30€ more per month)  

- **Men's salaries have a slightly wider range**. While the maximum wage is identical for both groups (3672€), the lowest salary in the dataset (673€) belongs to a male employee

- **Male salaries are more heavily concentrated in lower wage brackets**. This is visible from a larger gap between the mean and the median for men (167,61€) than for women (143,94€), alongside the lower minimum pay

- **For both genders, high earners push up the average monthly salary**. Because the mean is higher than the median for both groups, a small number of higher salaries pull the averages upward, skewing the distribution, making the typical worker's pay closer to the median than the mean


### **Table 2:** summarizing **average wage by gender and by country of origin**  
('salari_mensual' and 'genere' in the rows, "pais_d'origen" in the columns):   


- *amb el salari mig per gènere (files) i país d'origen (columnes).*  
- *Afegeix-hi les mitjanes als marges de la taula.*  


In [157]:
# creating and saving the table:
wage_gender_country = pd.crosstab(df_final['genere'], df["pais_d'origen"], df['salari_mensual'], aggfunc = 'mean', margins = True)
display(wage_gender_country)

pais_d'origen,Alemanya,Espanya,França,Itàlia,Noruega,All
genere,,,,,,
D,1527.70,1591.02,1437.44,1521.83,1816.19,1577.45
H,1854.09,1539.79,1562.35,1586.20,1346.00,1547.61
All,1602.50,1564.50,1496.99,1554.61,1641.23,1562.74


- Aplica format condicional a la taula per veure en un color més intens els valors més elevats.

In [158]:
# displaying the saved table with conditional formatting (3-colour gradient) using The Styler, and limiting decimals shown
wage_gender_country.style.background_gradient(cmap ='YlGnBu', axis = None).format(precision=2)

pais_d'origen,Alemanya,Espanya,França,Itàlia,Noruega,All
genere,,,,,,
D,1527.70,1591.02,1437.44,1521.83,1816.19,1577.45
H,1854.09,1539.79,1562.35,1586.20,1346.00,1547.61
All,1602.50,1564.50,1496.99,1554.61,1641.23,1562.74


From the above summary table depicting average wages per gender, and incorporating a 3rd dimension - country of origin - into the analysis, we can see that:  

- Eventhough  overall the statistics state that women have a higher average wage than the male workers (as seen in Table 1, women 1577€ and men 1547€), when the 3rd dimension of country is included in the study, we can see that, **per country, women have a lower average wage than their fellow male workers in 3 of the 5 countries** included in the dataset (Germany, France and Italy).  

- If we look at average salaries for female employees in comparison to the **overall company average salary** (1562€), the **average salary for women is below this in 4 out of the 5 countries** in the dataset (only Spain, at 1591€, has an average women's salary above the company-wide average)  

- **Norway** is the country that has the **highest average salary for female workers** (1816€), which is the 2nd highest average salary out of all the countries and gender groups in the dataset (1st place is taken by male employees in Germany, averaging at 1854€ monthly salary).  

- **Norway** is also the country where **overall average wages are the highest** (1641€, followed by Germany at 1602€).  

- However, in **Norway the average male employee salary is the lowest** out of all the countries and gender groups (1346€). This suggests that:
  1) a majority of male employees in Norway are at the lower salary levels
  2) the average female and overall salaries for the country may be distorted (pushed-up) by **a few high-earning female employees working for the company in Norway**  
  3) or, that the workforce in **Norway is a very small number of employees**, meaning that any single high or low salary has greater impact on the country's averages

- For **Spain**, the **average overall salary** (1564€) is **almost identical to the company-wide average (1562€)**, *suggesting* that the employees from Spain represent the majority of the 994 employees in the dataset, but without the actual employee-count data for each group this cannot be confrimed (it could just be a coincidence).  

- **Germany is the country with the largest gender wage gap** (men averaging at 1854€ salary, and women 327€ less at 1527€ average monthly salary)  




### (EXTRA)


*Abans d'interpretar mitjanes, medianes o totals, cal preguntar-se si aquests valors permeten una comparació justa entre grups.*  

*Quan les mides dels grups són diferents, els percentatges o ràtios solen oferir una visió més objectiva que els valors absoluts.*

*Podries **repetir l’anàlisi utilitzant percentatges i ràtios salarials**, i comparar com canvia la interpretació respecte als valors absoluts?*

As the size of each employee group is not accounted for in our summary table, and given the ambiguity surrounding the interpretation of the results from Norway and from Spain, for example, it is advisable to take another look at the data grouped by gender and country in terms of the percentage each group's salary represents in the dataset as a whole.  

In the following table, the same data has `sum` applied (instead of `mean`) and is **converted into percentages**, which gives us the **percentage of the total company salary budget each group receives**:

In [159]:
pc_wage_gender_country = pd.crosstab(df_final['genere'], df["pais_d'origen"], df['salari_mensual'], aggfunc = 'sum', margins = True, normalize = 'all')*100

# apply the same format as above, but rounding up or down the percentages to the nearest whole number
pc_wage_gender_country.style.background_gradient(cmap ='YlGnBu', axis = None).format(precision=0)

pais_d'origen,Alemanya,Espanya,França,Itàlia,Noruega,All
genere,,,,,,
D,4,35,4,5,3,51
H,1,36,4,6,1,49
All,5,71,8,11,5,100


Now in the above table we can see that:  

- **Spain does account for over 70% of the employee budget** in our dataset, confriming  the interpretation of the proximity between the country average salary amount and the global dataset average amount.  

- Although female workers in **Spain** earn a higher average monthly salary than their male colleagues (as we saw above in table 2, €1,591 women vs. €1,540 men), the **men in Spain have a slightly larger share of the total salary budget** (36.38% vs. 35.03%). This contradiction indicates a **higher overall population of male employees** in the Spanish workforce represented in the dataset.

- **Germany and Norway have the smallest workforces in the company (5% each of the total employee budget)**, which explains their "top positions" as described in the above interpretation (largest gender wage gap, lowest average male wage, highest global average wage, etc) as, being both small groups within the whole, any data point at the upper or lower ends of the general ranges **skews their averages substantially**.

In order to confirm the above suppositions, on **size of workforce in each country** as interpretted from the percentage share of the total salary budget, we can create another table (using pivot_table instead of crosstab, in order to calculate more than one aggregate value) which also shows the employee count for each group of country and gender to the right of the average salary values calculated previously.

In [171]:
# Create pivot table with 2 aggregates applied to the groups: mean salary and employee count
pivot_mean_and_count = pd.pivot_table(
    df_final,
    values='salari_mensual',
    index='genere',
    columns="pais_d'origen",
    aggfunc=['mean', 'count'],
    margins=True,
    margins_name='All'
)

pivot_mean_and_count.style.background_gradient(cmap ='YlGnBu', axis = None).format(precision=0)

In [160]:
pc_wage_gender_country

pais_d'origen,Alemanya,Espanya,França,Itàlia,Noruega,All
genere,,,,,,
D,3.64,35.03,4.16,5.19,3.16,51.18
H,1.31,36.38,4.12,5.62,1.39,48.82
All,4.95,71.41,8.29,10.81,4.54,100.00
